In [328]:
# ============================================================
# CELL 1 — IMPORT REQUIRED LIBRARIES
# ============================================================
#
# PURPOSE:
#
# Import all Python libraries required for:
#
# - financial time-series manipulation
# - mathematical calculations
# - downloading historical prices
# - CAPM regression
# - interactive Plotly visualisations
#
# IMPORTANT:
#
# ALL graphs in this notebook use Plotly.
#
# matplotlib is NOT used.
#
# ============================================================


# ------------------------------------------------------------
# PANDAS
# ------------------------------------------------------------

# pandas is used to:
#
# - store financial data
# - align observations by date
# - calculate returns
# - calculate rolling statistics
# - rank stocks cross-sectionally
# - create performance tables
#
# "pd" is the conventional abbreviation for pandas.
import pandas as pd


# ------------------------------------------------------------
# NUMPY
# ------------------------------------------------------------

# NumPy is used for numerical calculations.
#
# Examples:
#
# np.sqrt(252)
#
# -> annualising volatility and Sharpe ratio
#
#
# np.prod(...)
#
# -> compounding returns
#
#
# np.sign(...)
#
# -> determining whether a signal is positive or negative
#
#
# np.select(...)
#
# -> constructing Alpha 10's momentum-direction rule
import numpy as np


# ------------------------------------------------------------
# YFINANCE
# ------------------------------------------------------------

# yfinance downloads historical financial data from
# Yahoo Finance.
#
# We use it to obtain:
#
# - our selected stock
# - a cross-sectional stock universe
# - SPY
# - ^IRX
#
# ASSUMPTION:
#
# Yahoo Finance is sufficiently accurate for an academic
# backtest.
#
# LIMITATION:
#
# Institutional research would normally use higher-quality
# point-in-time datasets such as CRSP, Bloomberg or Refinitiv.
import yfinance as yf


# ------------------------------------------------------------
# STATSMODELS
# ------------------------------------------------------------

# statsmodels is used for the CAPM regression:
#
# Rp - Rf
# =
# alpha
# +
# beta(Rm - Rf)
# +
# epsilon
#
# where:
#
# Rp    = strategy return
# Rf    = risk-free return
# Rm    = market return
# alpha = CAPM abnormal return
# beta  = market sensitivity
import statsmodels.api as sm


# ------------------------------------------------------------
# PLOTLY
# ------------------------------------------------------------

# Plotly Graph Objects creates interactive graphs.
#
# The graphs allow us to:
#
# - zoom
# - hover over dates
# - inspect exact values
# - hide/show traces
#
# "go" is the conventional abbreviation for:
#
# plotly.graph_objects
import plotly.graph_objects as go

In [329]:
# ============================================================
# CELL 2 — DEFINE BACKTEST PARAMETERS
# ============================================================
#
# PURPOSE:
#
# Define:
#
# - stock to trade
# - signal universe
# - market benchmark
# - risk-free proxy
# - dates
# - rolling-window assumptions
# - trading costs
#
#
# NEW STRATEGY:
#
#                 ALPHA 10
#                     +
#                 ALPHA 101
#                     ↓
#              REGIME DETECTOR
#                     ↓
#        ┌────────────┴────────────┐
#        ↓                         ↓
# MEAN REVERSION               MOMENTUM
#        ↓                         ↓
# Alpha 42                    Alpha 10
#    +                            +
# Alpha 43                    Alpha 101
#        ↓                         ↓
# confirmation                confirmation
#        ↓                         ↓
# LONG / SHORT / CASH       LONG / SHORT / CASH
#
#
# ALL signals are observed using information available
# at today's CLOSE.
#
# Positions are entered at the NEXT trading day's OPEN.
#
# ============================================================


# ------------------------------------------------------------
# SELECT STOCK
# ------------------------------------------------------------

# THIS IS THE MAIN VARIABLE TO CHANGE.
#
# Examples:
#
# "MSFT"
# "NVDA"
# "CRWD"
# "AAPL"
# "META"
#
# Everything else in the notebook automatically uses this
# ticker in:
#
# - column names
# - graph titles
# - legends
# - hover labels
# - performance tables
# - benchmark descriptions
ASSET_TICKER = "NVDA"


# ------------------------------------------------------------
# MARKET BENCHMARK
# ------------------------------------------------------------

# SPY is used as the broad US equity-market proxy.
MARKET_TICKER = "SPY"


# ------------------------------------------------------------
# RISK-FREE PROXY
# ------------------------------------------------------------

# ^IRX is the 13-week US Treasury Bill yield.
RF_TICKER = "^IRX"


# ------------------------------------------------------------
# STRATEGY LABEL
# ------------------------------------------------------------

# Create a dynamic strategy name.
#
# If:
#
# ASSET_TICKER = "MSFT"
#
# this becomes:
#
# "MSFT Adaptive Alpha Regime Strategy"
STRATEGY_LABEL = (
    f"{ASSET_TICKER} Adaptive Alpha Regime Strategy"
)


# ------------------------------------------------------------
# DYNAMIC COLUMN NAMES
# ------------------------------------------------------------

# These variables ensure dataframe column names automatically
# change when ASSET_TICKER changes.

ASSET_OPEN_COL = (
    f"{ASSET_TICKER}_Open"
)

ASSET_HIGH_COL = (
    f"{ASSET_TICKER}_High"
)

ASSET_LOW_COL = (
    f"{ASSET_TICKER}_Low"
)

ASSET_CLOSE_COL = (
    f"{ASSET_TICKER}_Close"
)

ASSET_VOLUME_COL = (
    f"{ASSET_TICKER}_Volume"
)

ASSET_VWAP_COL = (
    f"{ASSET_TICKER}_VWAP_Proxy"
)

ASSET_RETURN_COL = (
    f"{ASSET_TICKER}_Return"
)


# Alpha diagnostic columns.
ALPHA10_COL = (
    f"{ASSET_TICKER}_Alpha10"
)

ALPHA101_COL = (
    f"{ASSET_TICKER}_Alpha101"
)

ALPHA42_COL = (
    f"{ASSET_TICKER}_Alpha42_Score"
)

ALPHA43_COL = (
    f"{ASSET_TICKER}_Alpha43_Score"
)


# Market columns.
MARKET_OPEN_COL = (
    f"{MARKET_TICKER}_Open"
)

MARKET_RETURN_COL = (
    f"{MARKET_TICKER}_Return"
)


# ------------------------------------------------------------
# CROSS-SECTIONAL SIGNAL UNIVERSE
# ------------------------------------------------------------

# WHY DO WE NEED MULTIPLE STOCKS?
#
# Alpha 10 and Alpha 42 use:
#
# rank(...)
#
# which the original paper defines as CROSS-SECTIONAL ranking.
#
# Therefore:
#
# MSFT must be ranked against other stocks.
#
# Using only MSFT would make cross-sectional rank meaningless.
#
#
# This is a practical academic universe of liquid US stocks.
#
# IMPORTANT LIMITATION:
#
# This is NOT a point-in-time historical index universe.
#
# Therefore survivorship bias can remain.
BASE_SIGNAL_UNIVERSE = [

    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "META",
    "GOOGL",
    "AVGO",
    "JPM",
    "XOM",
    "JNJ",
    "PG",
    "KO",
    "PEP",
    "COST",
    "WMT",
    "HD",
    "V",
    "MA",
    "UNH",
    "LLY",
    "ORCL",
    "CRM",
    "AMD",
    "QCOM",
    "NFLX"
]


# Add ASSET_TICKER automatically.
#
# dict.fromkeys(...) removes duplicates while preserving order.
#
# Therefore if:
#
# ASSET_TICKER = "MSFT"
#
# MSFT will NOT appear twice.
SIGNAL_UNIVERSE = list(
    dict.fromkeys(
        [ASSET_TICKER]
        +
        BASE_SIGNAL_UNIVERSE
    )
)


# ------------------------------------------------------------
# BACKTEST DATES
# ------------------------------------------------------------

# Fixed dates keep the experiment reproducible.
START_DATE = "2019-01-01"


# yfinance treats END_DATE as exclusive.
END_DATE = "2026-09-05"


# ------------------------------------------------------------
# ALPHA 10 PARAMETER
# ------------------------------------------------------------

# Alpha 10 checks whether recent Close changes have remained
# consistently positive or consistently negative.
#
# The original formula uses 4 observations.
ALPHA10_WINDOW = 4


# ------------------------------------------------------------
# ALPHA 43 PARAMETERS
# ------------------------------------------------------------

# Alpha 43 uses a 7-day Close change.
ALPHA43_PRICE_LOOKBACK = 7


# Alpha 43 uses ADV20.
ADV_WINDOW = 20


# ------------------------------------------------------------
# ROLLING PERFORMANCE WINDOW
# ------------------------------------------------------------

# Approximately one trading year.
ROLLING_WINDOW = 252


# ------------------------------------------------------------
# ANNUALISATION
# ------------------------------------------------------------

TRADING_DAYS = 252


# ------------------------------------------------------------
# TRANSACTION COST
# ------------------------------------------------------------

# Transaction cost per unit of position change.
#
# Example:
#
# 5 bps
# =
# 0.05%
#
# Set to zero for the initial academic test.
TRANSACTION_COST_BPS = 0.0


# ------------------------------------------------------------
# SHORT-BORROW COST
# ------------------------------------------------------------

# Because the new strategy can SHORT the stock,
# short-borrow costs can matter.
#
# Set to zero initially.
#
# Later we can test:
#
# 100 bps = 1% per year
#
# or more for hard-to-borrow securities.
SHORT_BORROW_BPS_ANNUAL = 0.0


# ------------------------------------------------------------
# DISPLAY SETTINGS
# ------------------------------------------------------------

print(
    "Selected stock       :",
    ASSET_TICKER
)

print(
    "Strategy             :",
    STRATEGY_LABEL
)

print(
    "Signal universe size :",
    len(SIGNAL_UNIVERSE)
)

print(
    "Market benchmark     :",
    MARKET_TICKER
)

print(
    "Risk-free proxy      :",
    RF_TICKER
)

Selected stock       : NVDA
Strategy             : NVDA Adaptive Alpha Regime Strategy
Signal universe size : 25
Market benchmark     : SPY
Risk-free proxy      : ^IRX


In [330]:
# ============================================================
# CELL 3 — DOWNLOAD HISTORICAL DATA
# ============================================================
#
# PURPOSE:
#
# Download:
#
# 1. the full signal universe
# 2. SPY
# 3. ^IRX
#
# The multiple-stock universe is required because several
# WorldQuant alpha formulas use cross-sectional ranking.
#
# ============================================================


# ------------------------------------------------------------
# DOWNLOAD SIGNAL UNIVERSE
# ------------------------------------------------------------

# Download OHLCV data for every stock used to calculate
# Alpha 10, Alpha 42 and Alpha 43.
#
# auto_adjust=True:
#
# adjusts OHLC values for corporate actions such as
# stock splits and distributions.
signal_raw = yf.download(

    SIGNAL_UNIVERSE,

    start=START_DATE,

    end=END_DATE,

    auto_adjust=True,

    progress=False,

    group_by="column"
)


# ------------------------------------------------------------
# DOWNLOAD MARKET BENCHMARK
# ------------------------------------------------------------

market_raw = yf.download(

    MARKET_TICKER,

    start=START_DATE,

    end=END_DATE,

    auto_adjust=True,

    progress=False
)


# ------------------------------------------------------------
# DOWNLOAD RISK-FREE YIELD
# ------------------------------------------------------------

rf_raw = yf.download(

    RF_TICKER,

    start=START_DATE,

    end=END_DATE,

    auto_adjust=False,

    progress=False
)


# ------------------------------------------------------------
# BASIC VALIDATION
# ------------------------------------------------------------

print(
    "Signal-universe date rows:",
    len(signal_raw)
)

print(
    f"{MARKET_TICKER} observations:",
    len(market_raw)
)

print(
    "Risk-free observations:",
    len(rf_raw)
)

Signal-universe date rows: 1930
SPY observations: 1930
Risk-free observations: 1930


In [331]:
# ============================================================
# CELL 4 — CLEAN AND ALIGN MARKET DATA
# ============================================================
#
# PURPOSE:
#
# Extract:
#
# Open
# High
# Low
# Close
# Volume
#
# for every stock in SIGNAL_UNIVERSE.
#
# We also create the main dataframe df used by the rest
# of the notebook.
#
# ============================================================


# ------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------

def extract_field(raw, field, tickers):

    """
    Extract one yfinance field such as Open or Close
    into a DataFrame where:

        rows    = dates
        columns = ticker symbols
    """


    # Convert tickers to list explicitly.
    tickers = list(tickers)


    # yfinance normally returns MultiIndex columns when
    # multiple tickers are downloaded.
    if isinstance(
        raw.columns,
        pd.MultiIndex
    ):


        # Common layout:
        #
        # Level 0:
        # Open / High / Low / Close / Volume
        #
        # Level 1:
        # ticker
        if field in raw.columns.get_level_values(0):

            output = (
                raw[field]
                .copy()
            )


        # Alternative layout:
        #
        # Level 0:
        # ticker
        #
        # Level 1:
        # OHLC field
        elif field in raw.columns.get_level_values(1):

            output = (
                raw.xs(
                    field,
                    axis=1,
                    level=1
                )
                .copy()
            )


        else:

            raise KeyError(
                f"Could not find {field} "
                "in downloaded data."
            )


    else:

        # Single-level-column fallback.
        output = (
            raw[[field]]
            .copy()
        )


        # If there is only one ticker,
        # give the column the actual ticker name.
        if len(tickers) == 1:

            output.columns = tickers


    # Ensure stock columns follow our desired universe order.
    return output.reindex(
        columns=tickers
    )


# ------------------------------------------------------------
# EXTRACT CROSS-SECTIONAL PANELS
# ------------------------------------------------------------

OPEN_PANEL = extract_field(
    signal_raw,
    "Open",
    SIGNAL_UNIVERSE
)


HIGH_PANEL = extract_field(
    signal_raw,
    "High",
    SIGNAL_UNIVERSE
)


LOW_PANEL = extract_field(
    signal_raw,
    "Low",
    SIGNAL_UNIVERSE
)


CLOSE_PANEL = extract_field(
    signal_raw,
    "Close",
    SIGNAL_UNIVERSE
)


VOLUME_PANEL = extract_field(
    signal_raw,
    "Volume",
    SIGNAL_UNIVERSE
)


# ------------------------------------------------------------
# EXTRACT SELECTED STOCK
# ------------------------------------------------------------

asset_open = (
    OPEN_PANEL[
        ASSET_TICKER
    ]
)


asset_high = (
    HIGH_PANEL[
        ASSET_TICKER
    ]
)


asset_low = (
    LOW_PANEL[
        ASSET_TICKER
    ]
)


asset_close = (
    CLOSE_PANEL[
        ASSET_TICKER
    ]
)


asset_volume = (
    VOLUME_PANEL[
        ASSET_TICKER
    ]
)


# ------------------------------------------------------------
# MARKET OPEN
# ------------------------------------------------------------

market_open = (
    market_raw[
        "Open"
    ]
    .squeeze()
)


# ------------------------------------------------------------
# RISK-FREE YIELD
# ------------------------------------------------------------

rf_yield = (
    rf_raw[
        "Close"
    ]
    .squeeze()
)


# ------------------------------------------------------------
# COMMON DATES
# ------------------------------------------------------------

# Keep only dates where:
#
# - selected stock traded
# - SPY traded
#
# We do NOT require every peer stock to have data.
#
# pandas ranking automatically ignores unavailable peers.
common_dates = (
    asset_open
    .dropna()
    .index
    .intersection(
        market_open.dropna().index
    )
)


# ------------------------------------------------------------
# ALIGN PANELS
# ------------------------------------------------------------

OPEN_PANEL = (
    OPEN_PANEL
    .reindex(
        common_dates
    )
)

HIGH_PANEL = (
    HIGH_PANEL
    .reindex(
        common_dates
    )
)

LOW_PANEL = (
    LOW_PANEL
    .reindex(
        common_dates
    )
)

CLOSE_PANEL = (
    CLOSE_PANEL
    .reindex(
        common_dates
    )
)

VOLUME_PANEL = (
    VOLUME_PANEL
    .reindex(
        common_dates
    )
)


# ------------------------------------------------------------
# CREATE MAIN DATAFRAME
# ------------------------------------------------------------

df = pd.DataFrame(
    index=common_dates
)


# Selected stock data.
df[ASSET_OPEN_COL] = (
    asset_open.reindex(
        common_dates
    )
)

df[ASSET_HIGH_COL] = (
    asset_high.reindex(
        common_dates
    )
)

df[ASSET_LOW_COL] = (
    asset_low.reindex(
        common_dates
    )
)

df[ASSET_CLOSE_COL] = (
    asset_close.reindex(
        common_dates
    )
)

df[ASSET_VOLUME_COL] = (
    asset_volume.reindex(
        common_dates
    )
)


# Market opening price.
df[MARKET_OPEN_COL] = (
    market_open.reindex(
        common_dates
    )
)


# Treasury yield.
#
# ffill():
#
# use the most recently available yield if a particular
# equity trading date has no new Treasury observation.
df["RF_Yield"] = (

    rf_yield
    .reindex(
        common_dates
    )
    .ffill()
)


# Remove rows where the selected asset, market or RF yield
# is unavailable.
df = df.dropna(
    subset=[

        ASSET_OPEN_COL,
        ASSET_HIGH_COL,
        ASSET_LOW_COL,
        ASSET_CLOSE_COL,
        ASSET_VOLUME_COL,
        MARKET_OPEN_COL,
        "RF_Yield"
    ]
)


print(
    df.head()
)


print(
    "\nFirst date:",
    df.index.min()
)


print(
    "Last date :",
    df.index.max()
)

            NVDA_Open  NVDA_High  NVDA_Low  NVDA_Close  NVDA_Volume  \
Date                                                                  
2019-01-02   3.234881   3.429014  3.220272    3.373052    508752000   
2019-01-03   3.312881   3.346804  3.161834    3.169262    705552000   
2019-01-04   3.242310   3.410443  3.211606    3.372310    585620000   
2019-01-07   3.429509   3.587737  3.378252    3.550842    709160000   
2019-01-08   3.632308   3.634537  3.389890    3.462442    786016000   

              SPY_Open  RF_Yield  
Date                              
2019-01-02  220.048737     2.365  
2019-01-03  222.061533     2.355  
2019-01-04  221.488959     2.358  
2019-01-07  226.051314     2.353  
2019-01-08  229.745978     2.400  

First date: 2019-01-02 00:00:00
Last date : 2026-09-04 00:00:00


In [332]:
# ============================================================
# CELL 5 — CREATE RETURNS AND OTHER REQUIRED INPUTS
# ============================================================


# ------------------------------------------------------------
# SELECTED STOCK OPEN-TO-OPEN RETURN
# ------------------------------------------------------------

# Return_t:
#
# Open_t / Open_(t-1) - 1
#
# This matches our holding interval:
#
# next day's open
# →
# following day's open
df[ASSET_RETURN_COL] = (

    df[ASSET_OPEN_COL]
    .pct_change(
        fill_method=None
    )
)


# ------------------------------------------------------------
# MARKET OPEN-TO-OPEN RETURN
# ------------------------------------------------------------

df[MARKET_RETURN_COL] = (

    df[MARKET_OPEN_COL]
    .pct_change(
        fill_method=None
    )
)


# ------------------------------------------------------------
# ANNUAL RISK-FREE RATE
# ------------------------------------------------------------

# ^IRX value:
#
# 4.50
#
# means approximately:
#
# 4.50%
df["RF_Annual"] = (

    df["RF_Yield"]
    /
    100
)


# ------------------------------------------------------------
# DAILY RISK-FREE RATE
# ------------------------------------------------------------

df["RF_Daily_Raw"] = (

    (
        1
        +
        df["RF_Annual"]
    )
    **
    (
        1 / TRADING_DAYS
    )
    -
    1
)


# ------------------------------------------------------------
# ALIGN RISK-FREE RATE
# ------------------------------------------------------------

# SIGNAL:
#
# Monday close
#
# ENTRY:
#
# Tuesday open
#
# RETURN:
#
# Tuesday open → Wednesday open
#
# Return appears on Wednesday row.
#
# Therefore Wednesday's strategy return must use the
# risk-free information that was known Monday.
#
# shift(2)
df["Rf_Daily"] = (

    df["RF_Daily_Raw"]
    .shift(2)
)


# ------------------------------------------------------------
# VWAP PROXY
# ------------------------------------------------------------

# True daily VWAP requires intraday price-volume data.
#
# Yahoo daily data does NOT provide true VWAP.
#
# We therefore use:
#
# Typical Price
# =
# (High + Low + Close) / 3
#
# IMPORTANT:
#
# This is a proxy.
#
# Alpha 42 should ideally be retested with true VWAP.
VWAP_PROXY_PANEL = (

    HIGH_PANEL
    +
    LOW_PANEL
    +
    CLOSE_PANEL

) / 3


# Add selected-stock proxy to main dataframe.
df[ASSET_VWAP_COL] = (

    VWAP_PROXY_PANEL[
        ASSET_TICKER
    ]
)


# ------------------------------------------------------------
# DOLLAR VOLUME
# ------------------------------------------------------------

DOLLAR_VOLUME_PANEL = (

    CLOSE_PANEL
    *
    VOLUME_PANEL
)


# ------------------------------------------------------------
# ADV20
# ------------------------------------------------------------

# ADV20:
#
# average daily dollar volume over 20 trading days.
ADV20_PANEL = (

    DOLLAR_VOLUME_PANEL
    .rolling(
        window=ADV_WINDOW,
        min_periods=ADV_WINDOW
    )
    .mean()
)


print(
    df[
        [
            ASSET_RETURN_COL,
            MARKET_RETURN_COL,
            "RF_Annual",
            "Rf_Daily",
            ASSET_VWAP_COL
        ]
    ].head(10)
)

            NVDA_Return  SPY_Return  RF_Annual  Rf_Daily  NVDA_VWAP_Proxy
Date                                                                     
2019-01-02          NaN         NaN    0.02365       NaN         3.340779
2019-01-03     0.024112    0.009147    0.02355       NaN         3.225967
2019-01-04    -0.021302   -0.002578    0.02358  0.000093         3.331453
2019-01-07     0.057736    0.020599    0.02353  0.000092         3.505610
2019-01-08     0.059133    0.016344    0.02400  0.000092         3.495623
2019-01-09    -0.032654    0.002881    0.02390  0.000092         3.523852
2019-01-10    -0.000705   -0.005047    0.02375  0.000094         3.550594
2019-01-11     0.017842    0.005541    0.02370  0.000094         3.646505
2019-01-14     0.016559   -0.003182    0.02360  0.000093         3.695038
2019-01-15     0.034351    0.003737    0.02383  0.000093         3.733666


In [333]:
# ============================================================
# CELL 6 — LONG-ONLY ADAPTIVE REGIME STRATEGY
# ============================================================
#
# PURPOSE:
#
# Determine whether the selected stock is currently in:
#
# 1. UPWARD MOMENTUM
# 2. DOWNWARD MOMENTUM
# 3. MEAN REVERSION
# 4. NEUTRAL
#
#
# IMPORTANT CHANGE:
#
# THE STRATEGY IS NOW LONG-ONLY.
#
# Therefore:
#
# +1 = WANT TO OWN THE STOCK
#
#  0 = DO NOT WANT TO OWN THE STOCK
#
# There is NO:
#
# -1 = SHORT
#
#
# ------------------------------------------------------------
# STRATEGY STRUCTURE
# ------------------------------------------------------------
#
#                  REGIME DETECTOR
#                        ↓
#
#       ┌────────────────┼─────────────────┐
#       ↓                ↓                 ↓
#   UP MOMENTUM     MEAN REVERSION      OTHER
#       ↓                ↓                 ↓
#     LONG         Alpha 42 + 43          CASH
#                   both LONG
#                     ↓
#                    LONG
#
#
# Downward Momentum:
#
# -> DO NOT SHORT
# -> request exit / remain in cash.
#
#
# Neutral:
#
# -> request exit / remain in cash.
#
#
# Mean-Reversion SHORT signal:
#
# -> IGNORE IT.
#
#
# ============================================================
# IMPORTANT EXIT RULE
# ============================================================
#
# Cell 6 only determines whether the strategy DESIRES
# to own the stock.
#
#
# Cell 7 decides whether the position can actually be closed.
#
#
# If:
#
# Desired_Long = 0
#
# but:
#
# current exit price < original entry price
#
# then:
#
# DO NOT CLOSE.
#
# Continue holding until the stock reaches its breakeven
# price.
#
# ============================================================



# ============================================================
# SECTION 1 — CROSS-SECTIONAL RANK
# ============================================================

def cross_sectional_rank(x):

    """
    Rank each stock against the other stocks in the
    SIGNAL_UNIVERSE on the same date.
    """


    # axis=1 means:
    #
    # rank ACROSS stocks.
    #
    # pct=True converts the ranking approximately into:
    #
    # 0 → lowest
    # 1 → highest
    return x.rank(
        axis=1,
        pct=True
    )



# ============================================================
# SECTION 2 — TIME-SERIES RANK
# ============================================================

def time_series_rank(x, window):

    """
    Rank today's observation relative to the same stock's
    own observations during the previous 'window' days.
    """


    # This function is applied inside every rolling window.
    def last_percentile_rank(a):


        # Convert the rolling NumPy array into a pandas Series.
        s = pd.Series(a)


        # If required data is missing:
        #
        # return NaN.
        if s.isna().any():

            return np.nan


        # Rank observations within the current window.
        ranked = s.rank(
            pct=True,
            method="average"
        )


        # Return only today's/latest observation.
        return ranked.iloc[-1]


    # Apply through time.
    return x.rolling(

        window=window,

        min_periods=window

    ).apply(

        last_percentile_rank,

        raw=True
    )



# ============================================================
# SECTION 3 — EXACT ALPHA 10
# ============================================================
#
# Alpha 10 looks at recent Close changes.
#
#
# If recent changes are consistently positive:
#
# follow the current price change.
#
#
# If recent changes are consistently negative:
#
# follow the current price change.
#
#
# Otherwise:
#
# reverse the current price change.
#
#
# The final result is ranked cross-sectionally.
#
# ============================================================


# Calculate:
#
# Close_t - Close_(t-1)
close_change_1d = (

    CLOSE_PANEL
    .diff(1)
)



# ------------------------------------------------------------
# ALL RECENT DAYS POSITIVE?
# ------------------------------------------------------------

alpha10_all_positive = (

    close_change_1d

    .rolling(

        window=ALPHA10_WINDOW,

        min_periods=ALPHA10_WINDOW
    )

    .min()

    >
    0
)



# ------------------------------------------------------------
# ALL RECENT DAYS NEGATIVE?
# ------------------------------------------------------------

alpha10_all_negative = (

    close_change_1d

    .rolling(

        window=ALPHA10_WINDOW,

        min_periods=ALPHA10_WINDOW
    )

    .max()

    <
    0
)



# ------------------------------------------------------------
# DEFAULT ALPHA 10 SIGNAL
# ------------------------------------------------------------
#
# Reverse today's price change.
alpha10_raw = (

    -close_change_1d
)



# ------------------------------------------------------------
# POSITIVE PERSISTENCE
# ------------------------------------------------------------

# If all recent days are positive:
#
# follow the actual Close change.
alpha10_raw = (

    alpha10_raw

    .where(

        ~alpha10_all_positive,

        close_change_1d
    )
)



# ------------------------------------------------------------
# NEGATIVE PERSISTENCE
# ------------------------------------------------------------

# If all recent days are negative:
#
# also follow the actual Close change.
alpha10_raw = (

    alpha10_raw

    .where(

        ~alpha10_all_negative,

        close_change_1d
    )
)



# ------------------------------------------------------------
# CROSS-SECTIONAL ALPHA 10
# ------------------------------------------------------------

alpha10_panel = (

    cross_sectional_rank(
        alpha10_raw
    )
)



# Save selected-stock Alpha 10.
df[ALPHA10_COL] = (

    alpha10_panel[
        ASSET_TICKER
    ]
)



# ============================================================
# SECTION 4 — ALPHA 101
# ============================================================
#
# Formula:
#
#              Close - Open
# Alpha101 = -----------------
#             High-Low+0.001
#
#
# Positive:
#
# upward intraday momentum.
#
#
# Negative:
#
# downward intraday momentum.
#
# ============================================================


alpha101_panel = (

    (
        CLOSE_PANEL
        -
        OPEN_PANEL
    )

    /

    (
        HIGH_PANEL
        -
        LOW_PANEL
        +
        0.001
    )
)



# Save selected stock.
df[ALPHA101_COL] = (

    alpha101_panel[
        ASSET_TICKER
    ]
)



# ============================================================
# SECTION 5 — ALPHA-10-INSPIRED MOMENTUM PERSISTENCE
# ============================================================
#
# Exact Alpha 10 is preserved above.
#
#
# For REGIME detection only:
#
# require 3 out of the previous 4 days
#
# rather than requiring all 4 days.
#
#
# Example:
#
# + + - +
#
# still indicates upward persistence.
#
# ============================================================


# Convert daily Close changes into:
#
# +1
# -1
#  0
close_direction_panel = (

    np.sign(
        close_change_1d
    )
)



# Count positive days.
alpha10_positive_days_panel = (

    (
        close_direction_panel
        >
        0
    )

    .rolling(

        window=ALPHA10_WINDOW,

        min_periods=ALPHA10_WINDOW
    )

    .sum()
)



# Count negative days.
alpha10_negative_days_panel = (

    (
        close_direction_panel
        <
        0
    )

    .rolling(

        window=ALPHA10_WINDOW,

        min_periods=ALPHA10_WINDOW
    )

    .sum()
)



# ------------------------------------------------------------
# CREATE MOMENTUM DIRECTION
# ------------------------------------------------------------

alpha10_direction_array = np.select(

    [

        alpha10_positive_days_panel
        >=
        3,

        alpha10_negative_days_panel
        >=
        3
    ],

    [

        1.0,

        -1.0
    ],

    default=0.0
)



# Restore dates and ticker columns.
alpha10_direction_panel = pd.DataFrame(

    alpha10_direction_array,

    index=CLOSE_PANEL.index,

    columns=CLOSE_PANEL.columns
)



# Keep selected stock only.
alpha10_direction = (

    alpha10_direction_panel[
        ASSET_TICKER
    ]
)



# ============================================================
# SECTION 6 — ALPHA 101 STRENGTH
# ============================================================
#
# Keep magnitude rather than only using +/-1.
#
#
# Example:
#
# Alpha101 = +0.80
#
# represents stronger momentum than:
#
# Alpha101 = +0.02
#
# ============================================================


alpha101_strength = (

    df[ALPHA101_COL]

    .clip(
        lower=-1,
        upper=1
    )
)



# ============================================================
# SECTION 7 — REGIME PARAMETERS
# ============================================================


# Smooth Alpha signals over approximately one week.
ALPHA_CONFIRM_WINDOW = 5


# Approximately one trading month.
MOMENTUM_SHORT_WINDOW = 20


# Approximately three trading months.
MOMENTUM_LONG_WINDOW = 60


# Efficiency-ratio calculation window.
EFFICIENCY_WINDOW = 20


# Require ER >= 0.30 to ENTER momentum.
MOMENTUM_ER_ENTRY = 0.30


# Momentum can remain until ER drops below 0.20.
MOMENTUM_ER_EXIT = 0.20


# Mean reversion only allowed in genuinely choppy periods.
MEAN_REVERSION_ER_THRESHOLD = 0.20



# ============================================================
# SECTION 8 — SMOOTH ALPHA 10
# ============================================================


alpha10_confirm = (

    alpha10_direction

    .rolling(

        window=ALPHA_CONFIRM_WINDOW,

        min_periods=ALPHA_CONFIRM_WINDOW
    )

    .mean()
)



# ============================================================
# SECTION 9 — SMOOTH ALPHA 101
# ============================================================


alpha101_confirm = (

    alpha101_strength

    .rolling(

        window=ALPHA_CONFIRM_WINDOW,

        min_periods=ALPHA_CONFIRM_WINDOW
    )

    .mean()
)



# ============================================================
# SECTION 10 — ALPHA MOMENTUM CONFIRMATION
# ============================================================
#
# Equal weighting:
#
# 50% Alpha-10 persistence
#
# +
#
# 50% Alpha-101 intraday momentum.
#
# ============================================================


df["Alpha_Momentum_Confirmation"] = (

    0.50
    *
    alpha10_confirm

    +

    0.50
    *
    alpha101_confirm
)



# ============================================================
# SECTION 11 — 20-DAY PRICE TREND
# ============================================================


df["Trend_20D_Return"] = (

    df[ASSET_CLOSE_COL]

    /

    df[ASSET_CLOSE_COL]
    .shift(
        MOMENTUM_SHORT_WINDOW
    )

    -

    1
)



# ============================================================
# SECTION 12 — 60-DAY PRICE TREND
# ============================================================


df["Trend_60D_Return"] = (

    df[ASSET_CLOSE_COL]

    /

    df[ASSET_CLOSE_COL]
    .shift(
        MOMENTUM_LONG_WINDOW
    )

    -

    1
)



# ============================================================
# SECTION 13 — EFFICIENCY RATIO
# ============================================================
#
# Formula:
#
#       |Close_t - Close_t-N|
# ER = --------------------------
#       Sum(|daily movements|)
#
#
# ER near 1:
#
# directional / trending.
#
#
# ER near 0:
#
# choppy / inefficient.
#
# ============================================================


# Net distance travelled.
trend_distance = (

    (
        df[ASSET_CLOSE_COL]

        -

        df[ASSET_CLOSE_COL]
        .shift(
            EFFICIENCY_WINDOW
        )
    )

    .abs()
)



# Absolute daily movement.
daily_absolute_move = (

    df[ASSET_CLOSE_COL]

    .diff()

    .abs()
)



# Total path travelled.
total_path_distance = (

    daily_absolute_move

    .rolling(

        window=EFFICIENCY_WINDOW,

        min_periods=EFFICIENCY_WINDOW
    )

    .sum()
)



# Final Efficiency Ratio.
df["Efficiency_Ratio"] = (

    trend_distance

    /

    total_path_distance.replace(

        0,

        np.nan
    )
)



# ============================================================
# SECTION 14 — MOMENTUM STATE MACHINE
# ============================================================
#
# Output:
#
# +1 = upward momentum
#
# -1 = downward momentum
#
#  0 = no established momentum.
#
#
# IMPORTANT:
#
# -1 here describes the MARKET REGIME.
#
# It does NOT mean we will short the stock.
#
# ============================================================


momentum_state = 0


momentum_state_series = pd.Series(

    0.0,

    index=df.index,

    dtype=float
)



# Process observations chronologically.
for date in df.index:


    # Today's Efficiency Ratio.
    er = df.loc[
        date,
        "Efficiency_Ratio"
    ]


    # Today's 20-day trend.
    trend20 = df.loc[
        date,
        "Trend_20D_Return"
    ]


    # Today's 60-day trend.
    trend60 = df.loc[
        date,
        "Trend_60D_Return"
    ]


    # Today's Alpha confirmation.
    alpha_confirm = df.loc[
        date,
        "Alpha_Momentum_Confirmation"
    ]



    # --------------------------------------------------------
    # MISSING DATA
    # --------------------------------------------------------

    if (

        pd.isna(er)

        or

        pd.isna(trend20)

        or

        pd.isna(trend60)

        or

        pd.isna(alpha_confirm)
    ):

        momentum_state = 0



    # --------------------------------------------------------
    # CURRENTLY NO MOMENTUM
    # --------------------------------------------------------

    elif momentum_state == 0:


        # Enter upward momentum.
        if (

            er
            >=
            MOMENTUM_ER_ENTRY

            and

            trend20
            >
            0

            and

            trend60
            >
            0

            and

            alpha_confirm
            >
            -0.10
        ):

            momentum_state = 1



        # Enter downward momentum.
        elif (

            er
            >=
            MOMENTUM_ER_ENTRY

            and

            trend20
            <
            0

            and

            trend60
            <
            0

            and

            alpha_confirm
            <
            0.10
        ):

            momentum_state = -1



    # --------------------------------------------------------
    # CURRENTLY UPWARD MOMENTUM
    # --------------------------------------------------------

    elif momentum_state == 1:


        # Strong reversal into downward momentum.
        if (

            er
            >=
            MOMENTUM_ER_ENTRY

            and

            trend20
            <
            0

            and

            trend60
            <
            0

            and

            alpha_confirm
            <
            0
        ):

            momentum_state = -1


        # Otherwise exit momentum if trend breaks down.
        elif (

            er
            <
            MOMENTUM_ER_EXIT

            or

            trend60
            <=
            0
        ):

            momentum_state = 0



    # --------------------------------------------------------
    # CURRENTLY DOWNWARD MOMENTUM
    # --------------------------------------------------------

    elif momentum_state == -1:


        # Strong reversal into upward momentum.
        if (

            er
            >=
            MOMENTUM_ER_ENTRY

            and

            trend20
            >
            0

            and

            trend60
            >
            0

            and

            alpha_confirm
            >
            0
        ):

            momentum_state = 1


        # Exit downward momentum.
        elif (

            er
            <
            MOMENTUM_ER_EXIT

            or

            trend60
            >=
            0
        ):

            momentum_state = 0



    # Save today's regime direction.
    momentum_state_series.loc[
        date
    ] = momentum_state



# Store in dataframe.
df["Momentum_Direction"] = (

    momentum_state_series
)



# ============================================================
# SECTION 15 — ALPHA 42
# ============================================================
#
# Mean-Reversion Signal #1.
#
#
# Formula:
#
# rank(VWAP-Close)
# ----------------
# rank(VWAP+Close)
#
#
# IMPORTANT:
#
# Our VWAP is a proxy:
#
# (High + Low + Close) / 3
#
# ============================================================


alpha42_panel = (

    cross_sectional_rank(

        VWAP_PROXY_PANEL
        -
        CLOSE_PANEL
    )

    /

    cross_sectional_rank(

        VWAP_PROXY_PANEL
        +
        CLOSE_PANEL
    )
)



# Centre score around zero.
alpha42_score_panel = (

    cross_sectional_rank(
        alpha42_panel
    )

    -

    0.5
)



# Store selected stock.
df[ALPHA42_COL] = (

    alpha42_score_panel[
        ASSET_TICKER
    ]
)



# ============================================================
# SECTION 16 — ALPHA 43
# ============================================================
#
# Mean-Reversion Signal #2.
#
#
# Uses:
#
# relative Volume
#
# +
#
# negative 7-day price movement.
#
# ============================================================


# Relative trading activity.
relative_volume_panel = (

    VOLUME_PANEL

    /

    ADV20_PANEL
)



# Rank relative volume within its own history.
alpha43_volume_component = (

    time_series_rank(

        relative_volume_panel,

        20
    )
)



# Negative 7-day Close change.
alpha43_negative_price_change = (

    -CLOSE_PANEL.diff(
        ALPHA43_PRICE_LOOKBACK
    )
)



# Rank reversal magnitude.
alpha43_price_component = (

    time_series_rank(

        alpha43_negative_price_change,

        8
    )
)



# Combine both Alpha 43 components.
alpha43_panel = (

    alpha43_volume_component

    *

    alpha43_price_component
)



# Centre around zero.
alpha43_score_panel = (

    cross_sectional_rank(
        alpha43_panel
    )

    -

    0.5
)



# Store selected stock.
df[ALPHA43_COL] = (

    alpha43_score_panel[
        ASSET_TICKER
    ]
)



# ============================================================
# SECTION 17 — MEAN-REVERSION DIRECTIONS
# ============================================================


alpha42_direction = (

    np.sign(
        df[ALPHA42_COL]
    )
)


alpha43_direction = (

    np.sign(
        df[ALPHA43_COL]
    )
)



# ============================================================
# SECTION 18 — LONG-ONLY MEAN-REVERSION SIGNAL
# ============================================================
#
# IMPORTANT CHANGE:
#
# We NEVER short.
#
#
# Alpha 42 positive
# AND
# Alpha 43 positive
#
# -> LONG signal.
#
#
# Any other combination:
#
# -> NO NEW LONG.
#
#
# In particular:
#
# Alpha 42 negative
# AND
# Alpha 43 negative
#
# previously meant SHORT.
#
# NOW:
#
# it simply means 0.
#
# ============================================================


mean_reversion_long_signal = (

    (
        alpha42_direction
        >
        0
    )

    &

    (
        alpha43_direction
        >
        0
    )

).astype(float)



# ============================================================
# SECTION 19 — CHECK SIGNAL AVAILABILITY
# ============================================================


signals_ready = (

    df[ALPHA10_COL]
    .notna()

    &

    df[ALPHA101_COL]
    .notna()

    &

    df[ALPHA42_COL]
    .notna()

    &

    df[ALPHA43_COL]
    .notna()

    &

    df["Alpha_Momentum_Confirmation"]
    .notna()

    &

    df["Trend_20D_Return"]
    .notna()

    &

    df["Trend_60D_Return"]
    .notna()

    &

    df["Efficiency_Ratio"]
    .notna()
)



# ============================================================
# SECTION 20 — TRUE MEAN-REVERSION REGIME
# ============================================================
#
# No established momentum
#
# AND
#
# ER < 0.20.
#
# ============================================================


true_mean_reversion = (

    signals_ready

    &

    (
        df["Momentum_Direction"]
        ==
        0
    )

    &

    (
        df["Efficiency_Ratio"]
        <
        MEAN_REVERSION_ER_THRESHOLD
    )
)



# ============================================================
# SECTION 21 — NEUTRAL REGIME
# ============================================================
#
# No established momentum
#
# BUT
#
# ER is not sufficiently low for mean reversion.
#
# ============================================================


neutral_regime = (

    signals_ready

    &

    (
        df["Momentum_Direction"]
        ==
        0
    )

    &

    (
        df["Efficiency_Ratio"]
        >=
        MEAN_REVERSION_ER_THRESHOLD
    )
)



# ============================================================
# SECTION 22 — CREATE REGIME LABELS
# ============================================================


df["Regime"] = pd.Series(

    np.nan,

    index=df.index,

    dtype="object"
)



# Upward Momentum.
df.loc[

    signals_ready

    &

    (
        df["Momentum_Direction"]
        ==
        1
    ),

    "Regime"

] = "Upward Momentum"



# Downward Momentum.
#
# IMPORTANT:
#
# This is only a REGIME label.
#
# We will NOT take a short position.
df.loc[

    signals_ready

    &

    (
        df["Momentum_Direction"]
        ==
        -1
    ),

    "Regime"

] = "Downward Momentum"



# Mean Reversion.
df.loc[

    true_mean_reversion,

    "Regime"

] = "Mean Reversion"



# Neutral.
df.loc[

    neutral_regime,

    "Regime"

] = "Neutral"



# ============================================================
# SECTION 23 — CREATE DESIRED LONG POSITION
# ============================================================
#
# Desired_Long:
#
# 1 = strategy wants to own the stock
#
# 0 = strategy wants to be out of the stock
#
#
# IMPORTANT:
#
# Desired_Long is NOT yet the actual position.
#
# Cell 7 will determine whether an existing position
# can actually be closed without realizing a loss.
#
# ============================================================


# Begin as NaN before signals are available.
df["Desired_Long"] = np.nan



# Once signals exist:
#
# default to not owning the stock.
df.loc[

    signals_ready,

    "Desired_Long"

] = 0.0



# ============================================================
# 23A — UPWARD MOMENTUM = LONG
# ============================================================


df.loc[

    signals_ready

    &

    (
        df["Momentum_Direction"]
        ==
        1
    ),

    "Desired_Long"

] = 1.0



# ============================================================
# 23B — MEAN REVERSION = LONG ONLY WHEN BOTH ALPHAS AGREE LONG
# ============================================================


df.loc[

    true_mean_reversion

    &

    (
        mean_reversion_long_signal
        ==
        1
    ),

    "Desired_Long"

] = 1.0



# ============================================================
# DOWNWARD MOMENTUM
# ============================================================
#
# Desired_Long remains:
#
# 0
#
# We DO NOT short.
#
# ============================================================



# ============================================================
# NEUTRAL
# ============================================================
#
# Desired_Long remains:
#
# 0
#
# ============================================================



# ============================================================
# NEGATIVE MEAN-REVERSION SIGNAL
# ============================================================
#
# Desired_Long remains:
#
# 0
#
# We DO NOT short.
#
# ============================================================



# ============================================================
# SECTION 24 — KEEP SIGNAL_RAW NAME FOR EXISTING NOTEBOOK
# ============================================================
#
# Your later notebook cells already refer to:
#
# Signal_Raw
#
# Therefore we retain that name.
#
#
# Signal_Raw now means:
#
# 1 = strategy would like to be LONG
#
# 0 = strategy would like to EXIT / remain CASH
#
# ============================================================


df["Signal_Raw"] = (

    df["Desired_Long"]
)



# ============================================================
# SECTION 25 — DIAGNOSTIC OUTPUT
# ============================================================


print(

    df[
        [
            ASSET_CLOSE_COL,

            ALPHA10_COL,

            ALPHA101_COL,

            "Alpha_Momentum_Confirmation",

            "Trend_20D_Return",

            "Trend_60D_Return",

            "Efficiency_Ratio",

            "Momentum_Direction",

            ALPHA42_COL,

            ALPHA43_COL,

            "Regime",

            "Desired_Long"
        ]
    ]

    .dropna(
        subset=[
            "Desired_Long"
        ]
    )

    .head(30)
)



# ============================================================
# SECTION 26 — REGIME SUMMARY
# ============================================================


print(
    "\nREGIME PERCENTAGES"
)


print(
    "------------------"
)


print(

    (
        df["Regime"]

        .value_counts(
            normalize=True
        )

        *
        100
    )

    .round(2)
)



# ============================================================
# SECTION 27 — DESIRED POSITION BY REGIME
# ============================================================
#
# Expected:
#
# Upward Momentum:
#
# mostly 1
#
#
# Downward Momentum:
#
# only 0
#
#
# Neutral:
#
# only 0
#
#
# Mean Reversion:
#
# either 0 or 1.
#
#
# There should NEVER be -1.
#
# ============================================================


print(
    "\nDESIRED LONG POSITION BY REGIME"
)


print(
    "-------------------------------"
)


print(

    pd.crosstab(

        df["Regime"],

        df["Desired_Long"]
    )
)

            NVDA_Close  NVDA_Alpha10  NVDA_Alpha101  \
Date                                                  
2019-03-29    4.450811          0.88      -0.142304   
2019-04-01    4.518231          0.68      -0.309114   
2019-04-02    4.536078          0.28      -0.075526   
2019-04-03    4.675383          0.44       0.580095   
2019-04-04    4.666460          0.44       0.060745   
2019-04-05    4.733138          0.64       0.300598   
2019-04-08    4.753959          0.40       0.506071   
2019-04-09    4.691249          0.32      -0.350113   
2019-04-10    4.761644          0.60       0.694857   
2019-04-11    4.747763          0.52      -0.166652   
2019-04-12    4.709837          0.80      -0.848820   
2019-04-15    4.578218          0.72      -0.730957   
2019-04-16    4.665224          0.40       0.476501   
2019-04-17    4.642416          0.52      -0.566863   
2019-04-18    4.617878          0.44      -0.332427   
2019-04-22    4.671668          0.60       0.616561   
2019-04-23

In [334]:
# ============================================================
# CELL 7 — LONG-ONLY EXECUTION WITH BREAKEVEN EXIT RULE
# ============================================================
#
# PURPOSE:
#
# Convert Cell 6's desired long signal into actual trades.
#
#
# Cell 6 produces:
#
# Desired_Long = 1
#
# -> strategy wants to own the stock.
#
#
# Desired_Long = 0
#
# -> strategy wants to close the stock / remain in cash.
#
#
# ============================================================
# IMPORTANT NEW RULE
# ============================================================
#
# IF THE STRATEGY WANTS TO EXIT:
#
# check whether closing would result in a loss relative to
# the original entry price.
#
#
# CASE 1:
#
# Exit price >= breakeven price
#
# -> CLOSE POSITION.
#
#
# CASE 2:
#
# Exit price < breakeven price
#
# -> DO NOT CLOSE.
#
# -> continue holding.
#
# -> mark the position as EXIT PENDING.
#
#
# Once the stock subsequently reaches the breakeven price:
#
# -> close the position.
#
#
# ============================================================
# NO SHORT SELLING
# ============================================================
#
# Position can ONLY be:
#
# 1 = LONG
#
# 0 = CASH
#
#
# Position can NEVER be:
#
# -1
#
# ============================================================



# ============================================================
# SECTION 1 — TRANSACTION-COST RATE
# ============================================================
#
# Example:
#
# TRANSACTION_COST_BPS = 5
#
# means:
#
# 5 / 10000
#
# =
#
# 0.0005
#
# =
#
# 0.05%.
#
# ============================================================


transaction_cost_rate = (

    TRANSACTION_COST_BPS

    /

    10000
)



# Protect against an invalid transaction-cost assumption.
if transaction_cost_rate >= 1:

    raise ValueError(

        "TRANSACTION_COST_BPS is unrealistically large."
    )



# ============================================================
# SECTION 2 — BREAKEVEN SETTINGS
# ============================================================
#
# If transaction costs are zero:
#
# breakeven price
#
# =
#
# original entry price.
#
#
# If transaction costs are positive:
#
# selling at exactly the entry price would still create
# a small economic loss because we paid costs on:
#
# BUY
#
# and:
#
# EXIT.
#
#
# Therefore we allow the breakeven calculation to include
# transaction costs.
#
# ============================================================


BREAKEVEN_INCLUDE_TRADING_COSTS = True



# Additional buffer above breakeven.
#
# Initially:
#
# 0 bps.
#
# Therefore we are NOT demanding an additional profit.
BREAKEVEN_BUFFER_BPS = 0.0



# Convert the buffer into decimal form.
breakeven_buffer_rate = (

    BREAKEVEN_BUFFER_BPS

    /

    10000
)



# ============================================================
# SECTION 3 — CREATE OUTPUT SERIES
# ============================================================
#
# These columns will store the results of the execution
# simulation.
#
# ============================================================


# Actual position:
#
# 1 = LONG
#
# 0 = CASH.
actual_position = pd.Series(

    0.0,

    index=df.index,

    dtype=float
)



# Store strategy returns.
strategy_return = pd.Series(

    np.nan,

    index=df.index,

    dtype=float
)



# Store original entry price while a position is open.
entry_price_series = pd.Series(

    np.nan,

    index=df.index,

    dtype=float
)



# Store breakeven price.
breakeven_price_series = pd.Series(

    np.nan,

    index=df.index,

    dtype=float
)



# True when:
#
# an exit has been requested
#
# BUT
#
# the stock is still below breakeven.
exit_pending_series = pd.Series(

    False,

    index=df.index,

    dtype=bool
)



# Store human-readable execution action.
trade_action_series = pd.Series(

    "CASH",

    index=df.index,

    dtype="object"
)



# Store regime responsible for current holding state.
held_regime_series = pd.Series(

    "Cash",

    index=df.index,

    dtype="object"
)



# Store exit execution price when a trade is closed.
exit_price_series = pd.Series(

    np.nan,

    index=df.index,

    dtype=float
)



# Store realized price return of completed trades.
realized_trade_return_series = pd.Series(

    np.nan,

    index=df.index,

    dtype=float
)



# Store unrealized price return of open positions.
unrealized_return_series = pd.Series(

    np.nan,

    index=df.index,

    dtype=float
)



# Record whether a BUY or EXIT occurred.
trade_series = pd.Series(

    0.0,

    index=df.index,

    dtype=float
)



# Store transaction-cost impact as a fraction of portfolio
# wealth on the execution date.
trading_cost_series = pd.Series(

    0.0,

    index=df.index,

    dtype=float
)



# ============================================================
# SECTION 4 — INITIAL PORTFOLIO STATE
# ============================================================
#
# Assume starting portfolio wealth:
#
# $1.
#
#
# This does NOT mean we literally invest only $1.
#
# It simply makes return calculations convenient.
#
# ============================================================


cash = 1.0



# Number of shares currently owned.
shares = 0.0



# Are we currently invested?
currently_long = False



# Has an exit been requested but blocked by the
# breakeven rule?
exit_pending = False



# Original purchase price.
entry_price = np.nan



# Required breakeven exit price.
breakeven_price = np.nan



# Regime that originally caused entry.
entry_regime = None



# ============================================================
# SECTION 5 — FIND FIRST VALID STRATEGY SIGNAL
# ============================================================
#
# Cell 6 requires historical observations before all
# indicators exist.
#
# Therefore early Desired_Long values are NaN.
#
# ============================================================


first_signal_date = (

    df["Desired_Long"]

    .first_valid_index()
)



# Make sure a signal exists.
if first_signal_date is None:

    raise ValueError(

        "No valid strategy signal was generated."
    )



# ============================================================
# SECTION 6 — EXECUTION LOOP
# ============================================================
#
# We process every trading date chronologically.
#
#
# IMPORTANT TIMING:
#
# Signal observed:
#
# previous day's CLOSE.
#
#
# Action:
#
# today's OPEN.
#
#
# This avoids using today's closing information
# before today's Open.
#
# ============================================================


for i in range(
    len(df)
):


    # Current trading date.
    date = df.index[i]



    # --------------------------------------------------------
    # TODAY'S OPEN PRICE
    # --------------------------------------------------------

    open_price = float(

        df.loc[
            date,
            ASSET_OPEN_COL
        ]
    )



    # --------------------------------------------------------
    # TODAY'S HIGH PRICE
    # --------------------------------------------------------
    #
    # This is used ONLY after an exit request has already
    # become active.
    #
    # If today's High reaches the breakeven price:
    #
    # we assume a resting sell-limit order at breakeven
    # could have been filled.
    high_price = float(

        df.loc[
            date,
            ASSET_HIGH_COL
        ]
    )



    # ========================================================
    # SIGNAL AVAILABLE AT TODAY'S OPEN
    # ========================================================
    #
    # At today's Open we only know YESTERDAY'S closing signal.
    #
    # Therefore:
    #
    # use Desired_Long from row i-1.
    #
    # This is the explicit event-driven equivalent of
    # lagging the signal.
    # ========================================================


    if i == 0:

        desired_long = 0.0

        previous_regime = None


    else:

        desired_long = (

            df["Desired_Long"]
            .iloc[
                i - 1
            ]
        )


        previous_regime = (

            df["Regime"]
            .iloc[
                i - 1
            ]
        )


        # Before strategy indicators exist:
        #
        # treat NaN as:
        #
        # no position.
        if pd.isna(
            desired_long
        ):

            desired_long = 0.0



    # Convert explicitly to float.
    desired_long = float(
        desired_long
    )



    # ========================================================
    # PORTFOLIO VALUE AT TODAY'S OPEN
    # ========================================================
    #
    # If currently long:
    #
    # market value =
    #
    # shares × today's Open.
    #
    #
    # If in cash:
    #
    # portfolio value = cash.
    #
    # ========================================================


    if currently_long:

        start_wealth = (

            shares
            *
            open_price
        )


    else:

        start_wealth = cash



    # Default values for today's action.
    action = (

        "HOLD LONG"

        if currently_long

        else

        "CASH"
    )



    # Assume no trade initially.
    traded_today = 0.0



    # Dollar transaction cost paid today.
    transaction_cost_amount = 0.0



    # This indicates whether cash is held for the entire
    # Open-to-next-Open period.
    #
    # If TRUE:
    #
    # cash earns the daily risk-free rate.
    cash_for_full_interval = False



    # ========================================================
    # SECTION 6A — CURRENTLY IN CASH
    # ========================================================


    if not currently_long:


        # ----------------------------------------------------
        # NEW BUY SIGNAL
        # ----------------------------------------------------

        if desired_long == 1.0:


            # Today's Open becomes our original entry price.
            entry_price = open_price



            # ------------------------------------------------
            # CALCULATE BREAKEVEN PRICE
            # ------------------------------------------------
            #
            # If transaction costs = 0:
            #
            # breakeven = entry price.
            #
            #
            # If costs > 0:
            #
            # price must rise slightly above entry to recover:
            #
            # - entry cost
            # - exit cost.
            if BREAKEVEN_INCLUDE_TRADING_COSTS:

                breakeven_price = (

                    entry_price

                    /

                    (
                        1
                        -
                        transaction_cost_rate
                    )
                    ** 2

                    *

                    (
                        1
                        +
                        breakeven_buffer_rate
                    )
                )


            else:

                breakeven_price = (

                    entry_price

                    *

                    (
                        1
                        +
                        breakeven_buffer_rate
                    )
                )



            # ------------------------------------------------
            # ENTRY TRANSACTION COST
            # ------------------------------------------------

            entry_cost = (

                cash

                *
                transaction_cost_rate
            )



            # Store transaction cost.
            transaction_cost_amount += (

                entry_cost
            )



            # Cash available after paying the entry cost.
            investable_cash = (

                cash

                -
                entry_cost
            )



            # Calculate number of shares purchased.
            shares = (

                investable_cash

                /

                entry_price
            )



            # All investable money is now in the stock.
            cash = 0.0



            # We are now LONG.
            currently_long = True



            # No exit is pending.
            exit_pending = False



            # Save the regime that caused the entry.
            entry_regime = (

                previous_regime

                if pd.notna(
                    previous_regime
                )

                else

                "Long Signal"
            )



            # Record action.
            action = "BUY AT OPEN"



            # One transaction occurred.
            traded_today = 1.0



        # ----------------------------------------------------
        # NO BUY SIGNAL
        # ----------------------------------------------------

        else:


            # Remain in cash.
            action = "CASH"


            # Because cash is held during the full interval:
            #
            # it earns Rf.
            cash_for_full_interval = True



    # ========================================================
    # SECTION 6B — CURRENTLY LONG
    # ========================================================

    else:


        # ====================================================
        # EXIT WAS ALREADY PENDING FROM AN EARLIER DAY
        # ====================================================

        if exit_pending:


            # ------------------------------------------------
            # CASE 1:
            # MARKET OPENS AT OR ABOVE BREAKEVEN
            # ------------------------------------------------
            #
            # We can exit immediately at today's Open.
            if (

                open_price
                >=
                breakeven_price
            ):


                # Save original entry price before resetting.
                original_entry_price = (

                    entry_price
                )



                # Gross sale proceeds.
                gross_exit_value = (

                    shares

                    *
                    open_price
                )



                # Exit transaction cost.
                exit_cost = (

                    gross_exit_value

                    *
                    transaction_cost_rate
                )



                transaction_cost_amount += (

                    exit_cost
                )



                # Convert stock into cash.
                cash = (

                    gross_exit_value

                    -
                    exit_cost
                )



                # Record exit price.
                exit_price_series.loc[
                    date
                ] = open_price



                # Price return before considering transaction
                # costs.
                realized_trade_return_series.loc[
                    date
                ] = (

                    open_price

                    /
                    original_entry_price

                    -
                    1
                )



                # No shares remain.
                shares = 0.0



                # Position is closed.
                currently_long = False



                # Exit request has been satisfied.
                exit_pending = False



                # Record action.
                action = (

                    "EXIT AT OPEN "
                    "(BREAKEVEN OR BETTER)"
                )



                traded_today = 1.0



                # Since exit occurred at the Open:
                #
                # cash is available for the full interval.
                cash_for_full_interval = True



                # Reset trade-specific variables.
                entry_price = np.nan

                breakeven_price = np.nan

                entry_regime = None



            # ------------------------------------------------
            # CASE 2:
            # STILL BELOW BREAKEVEN AT OPEN
            # ------------------------------------------------

            else:


                action = (

                    "BREAKEVEN HOLD"
                )



        # ====================================================
        # NO EXIT CURRENTLY PENDING
        # ====================================================

        else:


            # ------------------------------------------------
            # STRATEGY STILL WANTS TO BE LONG
            # ------------------------------------------------

            if desired_long == 1.0:


                # Keep holding.
                action = "HOLD LONG"



            # ------------------------------------------------
            # STRATEGY WANTS TO EXIT
            # ------------------------------------------------

            else:


                # --------------------------------------------
                # CAN EXIT WITHOUT A LOSS
                # --------------------------------------------

                if (

                    open_price
                    >=
                    breakeven_price
                ):


                    original_entry_price = (

                        entry_price
                    )



                    # Gross sale value.
                    gross_exit_value = (

                        shares

                        *
                        open_price
                    )



                    # Exit transaction cost.
                    exit_cost = (

                        gross_exit_value

                        *
                        transaction_cost_rate
                    )



                    transaction_cost_amount += (

                        exit_cost
                    )



                    # Convert position into cash.
                    cash = (

                        gross_exit_value

                        -
                        exit_cost
                    )



                    # Record exit execution price.
                    exit_price_series.loc[
                        date
                    ] = open_price



                    # Record trade's price return.
                    realized_trade_return_series.loc[
                        date
                    ] = (

                        open_price

                        /
                        original_entry_price

                        -
                        1
                    )



                    # No stock remains.
                    shares = 0.0



                    # Flat.
                    currently_long = False



                    # No pending exit.
                    exit_pending = False



                    action = (

                        "EXIT AT OPEN"
                    )



                    traded_today = 1.0



                    # We exited immediately at the Open.
                    cash_for_full_interval = True



                    # Reset entry information.
                    entry_price = np.nan

                    breakeven_price = np.nan

                    entry_regime = None



                # --------------------------------------------
                # EXIT WOULD REALIZE A LOSS
                # --------------------------------------------

                else:


                    # DO NOT SELL.
                    #
                    # Instead:
                    #
                    # keep holding until breakeven.
                    exit_pending = True


                    action = (

                        "EXIT BLOCKED - "
                        "HOLD TO BREAKEVEN"
                    )



    # ========================================================
    # SECTION 7 — CHECK BREAKEVEN LIMIT DURING TODAY
    # ========================================================
    #
    # This only applies if:
    #
    # 1. we are still LONG
    #
    # AND
    #
    # 2. an exit is pending
    #
    # AND
    #
    # 3. today's High reaches the required breakeven price.
    #
    #
    # ASSUMPTION:
    #
    # Once an exit becomes pending, a sell-limit order is
    # resting at the breakeven price.
    #
    #
    # If:
    #
    # High >= breakeven
    #
    # we assume the order could have filled.
    #
    # ========================================================


    if (

        currently_long

        and

        exit_pending

        and

        high_price
        >=
        breakeven_price
    ):


        # Save original entry before resetting.
        original_entry_price = (

            entry_price
        )



        # Exit exactly at the breakeven limit price.
        limit_exit_price = (

            breakeven_price
        )



        # Gross proceeds.
        gross_exit_value = (

            shares

            *
            limit_exit_price
        )



        # Exit transaction cost.
        exit_cost = (

            gross_exit_value

            *
            transaction_cost_rate
        )



        transaction_cost_amount += (

            exit_cost
        )



        # Convert shares into cash.
        cash = (

            gross_exit_value

            -
            exit_cost
        )



        # Record exit.
        exit_price_series.loc[
            date
        ] = limit_exit_price



        # Record price return.
        realized_trade_return_series.loc[
            date
        ] = (

            limit_exit_price

            /
            original_entry_price

            -
            1
        )



        # Position closed.
        shares = 0.0


        currently_long = False


        exit_pending = False



        # Record action.
        action = (

            "EXIT AT BREAKEVEN LIMIT"
        )



        traded_today = 1.0



        # Because exit occurred sometime intraday rather than
        # exactly at the Open:
        #
        # we conservatively do NOT credit a full day's
        # risk-free return for this interval.
        cash_for_full_interval = False



        # Reset trade information.
        entry_price = np.nan


        breakeven_price = np.nan


        entry_regime = None



    # ========================================================
    # SECTION 8 — RECORD TODAY'S ACTUAL POSITION
    # ========================================================


    actual_position.loc[
        date
    ] = (

        1.0

        if currently_long

        else

        0.0
    )



    # ========================================================
    # SECTION 9 — RECORD EXIT-PENDING STATUS
    # ========================================================


    exit_pending_series.loc[
        date
    ] = (

        exit_pending
    )



    # ========================================================
    # SECTION 10 — RECORD ENTRY / BREAKEVEN PRICE
    # ========================================================


    if currently_long:


        entry_price_series.loc[
            date
        ] = (

            entry_price
        )


        breakeven_price_series.loc[
            date
        ] = (

            breakeven_price
        )


        # Unrealized price P&L measured using today's Open.
        unrealized_return_series.loc[
            date
        ] = (

            open_price

            /
            entry_price

            -
            1
        )



    # ========================================================
    # SECTION 11 — RECORD HUMAN-READABLE STATE
    # ========================================================


    trade_action_series.loc[
        date
    ] = action



    if currently_long:


        if exit_pending:

            held_regime_series.loc[
                date
            ] = "Breakeven Hold"


        else:

            held_regime_series.loc[
                date
            ] = (

                entry_regime
            )


    else:

        held_regime_series.loc[
            date
        ] = "Cash"



    # ========================================================
    # SECTION 12 — RECORD TRADE AND COST
    # ========================================================


    trade_series.loc[
        date
    ] = traded_today



    if start_wealth > 0:

        trading_cost_series.loc[
            date
        ] = (

            transaction_cost_amount

            /
            start_wealth
        )



    # ========================================================
    # SECTION 13 — CALCULATE RETURN TO NEXT OPEN
    # ========================================================
    #
    # Asset_Return was defined as:
    #
    # Open_t / Open_(t-1) - 1
    #
    #
    # Therefore strategy return is also recorded using
    # approximately the Open-to-Open interval.
    #
    #
    # If LONG:
    #
    # next wealth is based on next day's Open.
    #
    #
    # If CASH for the entire interval:
    #
    # cash earns Rf.
    #
    #
    # If exited intraday at breakeven:
    #
    # we conservatively do not credit the entire day's Rf.
    #
    # ========================================================


    # Last row has no following Open.
    if i < (
        len(df)
        -
        1
    ):


        # Next trading date.
        next_date = (

            df.index[
                i + 1
            ]
        )



        # Next Open.
        next_open = float(

            df.loc[
                next_date,
                ASSET_OPEN_COL
            ]
        )



        # Risk-free rate associated with this holding interval.
        rf_period = (

            df["Rf_Daily"]
            .iloc[
                i + 1
            ]
        )



        # During early warm-up it may be NaN.
        if pd.isna(
            rf_period
        ):

            rf_period = 0.0


        else:

            rf_period = float(
                rf_period
            )



        # ----------------------------------------------------
        # STILL LONG
        # ----------------------------------------------------

        if currently_long:


            # Mark the position to next day's Open.
            end_wealth = (

                shares

                *
                next_open
            )



        # ----------------------------------------------------
        # FLAT / CASH
        # ----------------------------------------------------

        else:


            # If cash was held for the entire period:
            #
            # earn the risk-free return.
            if cash_for_full_interval:

                cash = (

                    cash

                    *
                    (
                        1
                        +
                        rf_period
                    )
                )



            # Ending wealth is cash.
            end_wealth = cash



        # ----------------------------------------------------
        # STRATEGY RETURN
        # ----------------------------------------------------

        strategy_return.loc[
            next_date
        ] = (

            end_wealth

            /
            start_wealth

            -
            1
        )



# ============================================================
# SECTION 14 — ADD EXECUTION RESULTS TO DF
# ============================================================


# Actual long/cash position.
df["Position"] = (

    actual_position
)



# Strategy daily return.
df["Strategy_Return"] = (

    strategy_return
)



# Original entry price.
df["Entry_Price"] = (

    entry_price_series
)



# Required exit price.
df["Breakeven_Price"] = (

    breakeven_price_series
)



# Is the strategy waiting for recovery?
df["Exit_Pending"] = (

    exit_pending_series
)



# Human-readable execution action.
df["Trade_Action"] = (

    trade_action_series
)



# Actual holding state.
df["Held_Regime"] = (

    held_regime_series
)



# Exit execution price.
df["Exit_Price"] = (

    exit_price_series
)



# Completed-trade return.
df["Realized_Trade_Return"] = (

    realized_trade_return_series
)



# Unrealized return while position remains open.
df["Unrealized_Return"] = (

    unrealized_return_series
)



# Trade indicator.
df["Trade"] = (

    trade_series
)



# Transaction-cost impact.
df["Trading_Cost"] = (

    trading_cost_series
)



# ============================================================
# SECTION 15 — COMPATIBILITY COLUMNS
# ============================================================
#
# No short position exists anymore.
#
# Therefore:
#
# Short_Borrow_Cost = 0.
#
#
# Cash weight:
#
# 1 when flat
#
# 0 when long.
#
# ============================================================


df["Short_Borrow_Cost"] = 0.0



df["Cash_Weight"] = (

    1.0

    -

    df["Position"]
)



# ============================================================
# SECTION 16 — RESTRICT TO EFFECTIVE BACKTEST PERIOD
# ============================================================
#
# First valid signal:
#
# generated at close.
#
#
# Entry:
#
# next Open.
#
#
# First comparable Open-to-Open return:
#
# following Open.
#
#
# Therefore the first strategy return is approximately
# two rows after the first valid signal.
#
# ============================================================


first_signal_location = (

    df.index.get_loc(
        first_signal_date
    )
)



# Calculate first valid return location.
first_return_location = (

    first_signal_location

    +
    2
)



# Ensure the date exists.
if first_return_location >= len(df):

    raise ValueError(

        "Not enough observations after the first strategy signal."
    )



# Determine actual date.
effective_start_date = (

    df.index[
        first_return_location
    ]
)



# Keep only the effective strategy period.
df = (

    df.loc[
        effective_start_date:
    ]

    .copy()
)



# Remove rows missing essential return data.
df = df.dropna(

    subset=[

        "Strategy_Return",

        ASSET_RETURN_COL,

        MARKET_RETURN_COL,

        "Rf_Daily"
    ]
)



# ============================================================
# SECTION 17 — CREATE STRATEGY WEALTH INDEX
# ============================================================
#
# This is important because an underwater trade that is
# refused permission to exit STILL suffers unrealized losses.
#
#
# Therefore:
#
# "I did not sell"
#
# does NOT mean:
#
# "I did not lose money."
#
#
# The Strategy_Return series continues to mark the stock
# to market while it is underwater.
#
# ============================================================


df["Strategy_Wealth"] = (

    (
        1
        +
        df["Strategy_Return"]
    )

    .cumprod()
)



# ============================================================
# SECTION 18 — VALIDATION:
# THERE MUST BE NO SHORT POSITIONS
# ============================================================


if (

    df["Position"]
    .min()

    <
    0
):

    raise ValueError(

        "ERROR: A short position was created in a long-only strategy."
    )



# ============================================================
# SECTION 19 — DISPLAY RESULTS
# ============================================================


print(
    "LONG-ONLY EXECUTION SAMPLE"
)


print(
    "--------------------------"
)


print(

    df[
        [
            "Regime",

            "Desired_Long",

            "Position",

            "Entry_Price",

            "Breakeven_Price",

            "Exit_Pending",

            "Trade_Action",

            "Unrealized_Return",

            "Realized_Trade_Return",

            "Strategy_Return"
        ]
    ]

    .head(30)
)



# ============================================================
# SECTION 20 — POSITION COUNTS
# ============================================================


print(
    "\nACTUAL POSITION COUNTS"
)


print(
    "----------------------"
)


print(

    df["Position"]
    .value_counts()
    .sort_index()
)



# ============================================================
# SECTION 21 — EXIT-PENDING COUNT
# ============================================================
#
# This tells us how many observations were spent holding
# a losing position because the strategy refused to realize
# the loss.
#
# ============================================================


print(
    "\nDAYS WAITING FOR BREAKEVEN"
)


print(
    "--------------------------"
)


print(

    int(
        df["Exit_Pending"]
        .sum()
    )
)



# ============================================================
# SECTION 22 — EFFECTIVE BACKTEST PERIOD
# ============================================================


print(
    "\nEffective strategy start:",
    df.index.min()
)


print(
    "Effective strategy end  :",
    df.index.max()
)

LONG-ONLY EXECUTION SAMPLE
--------------------------
                     Regime  Desired_Long  Position  Entry_Price  \
Date                                                               
2019-04-02  Upward Momentum           1.0       1.0     4.542276   
2019-04-03  Upward Momentum           1.0       1.0     4.542276   
2019-04-04  Upward Momentum           1.0       1.0     4.542276   
2019-04-05  Upward Momentum           1.0       1.0     4.542276   
2019-04-08  Upward Momentum           1.0       1.0     4.542276   
2019-04-09  Upward Momentum           1.0       1.0     4.542276   
2019-04-10  Upward Momentum           1.0       1.0     4.542276   
2019-04-11  Upward Momentum           1.0       1.0     4.542276   
2019-04-12  Upward Momentum           1.0       1.0     4.542276   
2019-04-15  Upward Momentum           1.0       1.0     4.542276   
2019-04-16  Upward Momentum           1.0       1.0     4.542276   
2019-04-17  Upward Momentum           1.0       1.0     4.5422

In [335]:
print(
    df["Regime"]
    .value_counts()
)


print(
    "\nREGIME PERCENTAGES"
)


print(
    (
        df["Regime"]
        .value_counts(
            normalize=True
        )
        * 100
    ).round(2)
)


print(
    "\nPOSITION BY REGIME"
)


print(
    pd.crosstab(
        df["Regime"],
        df["Signal_Raw"]
    )
)

Regime
Mean Reversion       915
Upward Momentum      528
Neutral              334
Downward Momentum     91
Name: count, dtype: int64

REGIME PERCENTAGES
Regime
Mean Reversion       48.98
Upward Momentum      28.27
Neutral              17.88
Downward Momentum     4.87
Name: proportion, dtype: float64

POSITION BY REGIME
Signal_Raw         0.0  1.0
Regime                     
Downward Momentum   91    0
Mean Reversion     503  412
Neutral            334    0
Upward Momentum      0  528


In [336]:
# ============================================================
# CELL 7A — PRICE, REGIME AND ACTUAL TRADE PLOT
# ============================================================
#
# PURPOSE:
#
# Display:
#
# 1. Selected stock Close
# 2. VWAP proxy
# 3. Upward Momentum periods
# 4. Downward Momentum periods
# 5. Mean-Reversion periods
# 6. Neutral periods
# 7. Actual BUY executions
# 8. Actual EXIT executions
#
# IMPORTANT:
#
# This is now consistent with the LONG-ONLY strategy.
#
# ============================================================


# Create the Plotly figure.
fig = go.Figure()


# ============================================================
# SELECTED STOCK CLOSING PRICE
# ============================================================

fig.add_trace(
    go.Scatter(

        # Trading dates.
        x=df.index,

        # Dynamic selected-stock Close.
        y=df[ASSET_CLOSE_COL],

        mode="lines",

        # Automatically becomes:
        #
        # MSFT Close
        # SOFI Close
        # NVDA Close
        # etc.
        name=f"{ASSET_TICKER} Close",

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{ASSET_TICKER} Close: $%{{y:.2f}}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# VWAP PROXY
# ============================================================

fig.add_trace(
    go.Scatter(

        x=df.index,

        y=df[ASSET_VWAP_COL],

        mode="lines",

        name=f"{ASSET_TICKER} VWAP Proxy",

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "VWAP Proxy: $%{y:.2f}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# UPWARD MOMENTUM
# ============================================================

up_dates = df.index[
    df["Regime"] == "Upward Momentum"
]


fig.add_trace(
    go.Scatter(

        x=up_dates,

        y=df.loc[
            up_dates,
            ASSET_CLOSE_COL
        ],

        mode="markers",

        name="Upward Momentum",

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{ASSET_TICKER}: $%{{y:.2f}}<br>"
            "Regime: Upward Momentum"
            "<extra></extra>"
        )
    )
)


# ============================================================
# DOWNWARD MOMENTUM
# ============================================================
#
# IMPORTANT:
#
# This is only a REGIME.
#
# The strategy DOES NOT short.
# ============================================================

down_dates = df.index[
    df["Regime"] == "Downward Momentum"
]


fig.add_trace(
    go.Scatter(

        x=down_dates,

        y=df.loc[
            down_dates,
            ASSET_CLOSE_COL
        ],

        mode="markers",

        name="Downward Momentum",

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{ASSET_TICKER}: $%{{y:.2f}}<br>"
            "Regime: Downward Momentum"
            "<extra></extra>"
        )
    )
)


# ============================================================
# MEAN REVERSION
# ============================================================

mr_dates = df.index[
    df["Regime"] == "Mean Reversion"
]


fig.add_trace(
    go.Scatter(

        x=mr_dates,

        y=df.loc[
            mr_dates,
            ASSET_CLOSE_COL
        ],

        mode="markers",

        name="Mean Reversion",

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{ASSET_TICKER}: $%{{y:.2f}}<br>"
            "Regime: Mean Reversion"
            "<extra></extra>"
        )
    )
)


# ============================================================
# NEUTRAL REGIME
# ============================================================

neutral_dates = df.index[
    df["Regime"] == "Neutral"
]


fig.add_trace(
    go.Scatter(

        x=neutral_dates,

        y=df.loc[
            neutral_dates,
            ASSET_CLOSE_COL
        ],

        mode="markers",

        name="Neutral",

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            f"{ASSET_TICKER}: $%{{y:.2f}}<br>"
            "Regime: Neutral"
            "<extra></extra>"
        )
    )
)


# ============================================================
# ACTUAL BUY EXECUTIONS
# ============================================================

buy_mask = (
    df["Trade_Action"]
    .astype(str)
    .str.startswith("BUY")
)


buy_dates = df.index[
    buy_mask
]


fig.add_trace(
    go.Scatter(

        x=buy_dates,

        y=df.loc[
            buy_dates,
            ASSET_OPEN_COL
        ],

        mode="markers",

        marker=dict(
            symbol="triangle-up",
            size=10
        ),

        name="Actual Buy",

        hovertemplate=(
            "BUY<br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Execution: $%{y:.2f}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# ACTUAL EXIT EXECUTIONS
# ============================================================

exit_mask = (
    df["Trade_Action"]
    .astype(str)
    .str.startswith("EXIT")
)


exit_dates = df.index[
    exit_mask
]


fig.add_trace(
    go.Scatter(

        x=exit_dates,

        y=df.loc[
            exit_dates,
            "Exit_Price"
        ],

        mode="markers",

        marker=dict(
            symbol="triangle-down",
            size=10
        ),

        name="Actual Exit",

        hovertemplate=(
            "EXIT<br>"
            "Date: %{x|%Y-%m-%d}<br>"
            "Execution: $%{y:.2f}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# FORMAT
# ============================================================

fig.update_layout(

    title=(
        f"{ASSET_TICKER} Price, "
        "Regime and Actual Long-Only Trades"
    ),

    xaxis_title="Date",

    yaxis_title=(
        f"{ASSET_TICKER} Price (USD)"
    ),

    template="plotly_white",

    hovermode="x unified",

    legend_title_text=""
)


fig.show()

In [337]:
# ============================================================
# CELL 7B — ACTUAL LONG-ONLY POSITION THROUGH TIME
# ============================================================
#
# Position:
#
# 1 = LONG
#
# 0 = CASH
#
# There is NO short position.
#
# ============================================================


fig = go.Figure()


# ============================================================
# ACTUAL POSITION
# ============================================================

fig.add_trace(
    go.Scatter(

        x=df.index,

        y=df["Position"],

        line_shape="hv",

        mode="lines",

        name="Actual Position",

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Position: %{y:.0f}"
            "<extra></extra>"
        )
    )
)


# ============================================================
# MARK DAYS WHERE EXIT IS PENDING
# ============================================================
#
# Exit_Pending = True means:
#
# the strategy wants to leave,
# BUT current price is below breakeven.
#
# Therefore it is forced to remain LONG.
# ============================================================

pending_dates = df.index[
    df["Exit_Pending"]
]


fig.add_trace(
    go.Scatter(

        x=pending_dates,

        y=np.ones(
            len(pending_dates)
        ),

        mode="markers",

        name="Holding for Breakeven",

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Exit requested but blocked<br>"
            "Waiting for breakeven"
            "<extra></extra>"
        )
    )
)


fig.update_layout(

    title=(
        f"{ASSET_TICKER} Long-Only "
        "Adaptive Strategy Position"
    ),

    xaxis_title="Date",

    yaxis_title="Portfolio Position",

    template="plotly_white",

    hovermode="x unified",

    legend_title_text=""
)


# Only two possible states now.
fig.update_yaxes(

    tickmode="array",

    tickvals=[
        0,
        1
    ],

    ticktext=[
        "Cash / Risk-Free",
        f"LONG {ASSET_TICKER}"
    ]
)


fig.show()

In [338]:
# ============================================================
# CELL 8 — CALCULATE EXCESS RETURNS FOR CAPM
# ============================================================
#
# CAPM:
#
# Rp - Rf
# =
# alpha
# +
# beta(Rm - Rf)
# +
# epsilon
#
# ============================================================


# Strategy excess return.
df["Strategy_Excess"] = (

    df["Strategy_Return"]

    -

    df["Rf_Daily"]
)


# Market excess return.
df["Market_Excess"] = (

    df[MARKET_RETURN_COL]

    -

    df["Rf_Daily"]
)


print(
    df[
        [
            "Strategy_Return",
            MARKET_RETURN_COL,
            "Rf_Daily",
            "Strategy_Excess",
            "Market_Excess"
        ]
    ].head()
)

            Strategy_Return  SPY_Return  Rf_Daily  Strategy_Excess  \
Date                                                                 
2019-04-02         0.000091    0.004707  0.000091         0.000000   
2019-04-03         0.009550    0.004475  0.000091         0.009459   
2019-04-04         0.016216   -0.001879  0.000093         0.016123   
2019-04-05         0.010638    0.003975  0.000093         0.010545   
2019-04-08        -0.001474    0.000625  0.000093        -0.001567   

            Market_Excess  
Date                       
2019-04-02       0.004616  
2019-04-03       0.004384  
2019-04-04      -0.001972  
2019-04-05       0.003882  
2019-04-08       0.000532  


In [339]:
# ============================================================
# CELL 9 — PERFORMANCE STATISTICS FUNCTION
# ============================================================


def performance_stats(
    returns,
    rf
):


    # --------------------------------------------------------
    # ALIGN RETURNS AND RISK-FREE RATE
    # --------------------------------------------------------

    data = pd.concat(
        [
            returns.rename("Return"),
            rf.rename("Rf")
        ],
        axis=1
    ).dropna()


    returns = data["Return"]

    rf = data["Rf"]


    # Number of observations.
    n = len(
        returns
    )


    if n == 0:

        return pd.Series({
            "Cumulative Return": np.nan,
            "Annualized Return": np.nan,
            "Annualized Volatility": np.nan,
            "Sharpe Ratio": np.nan,
            "Maximum Drawdown": np.nan
        })


    # --------------------------------------------------------
    # CUMULATIVE RETURN
    # --------------------------------------------------------

    cumulative_return = (

        (
            1
            +
            returns
        )
        .prod()

        -
        1
    )


    # --------------------------------------------------------
    # ANNUALISED RETURN
    # --------------------------------------------------------

    if (
        1
        +
        cumulative_return
    ) > 0:

        annualized_return = (

            (
                1
                +
                cumulative_return
            )

            **
            (
                TRADING_DAYS
                /
                n
            )

            -
            1
        )

    else:

        annualized_return = np.nan


    # --------------------------------------------------------
    # ANNUALISED VOLATILITY
    # --------------------------------------------------------

    annualized_volatility = (

        returns.std(
            ddof=1
        )

        *
        np.sqrt(
            TRADING_DAYS
        )
    )


    # --------------------------------------------------------
    # EXCESS RETURNS
    # --------------------------------------------------------

    excess_returns = (

        returns
        -
        rf
    )


    # Daily excess volatility.
    excess_std = (

        excess_returns
        .std(
            ddof=1
        )
    )


    # --------------------------------------------------------
    # SHARPE RATIO
    # --------------------------------------------------------

    if (
        pd.notna(
            excess_std
        )

        and

        excess_std
        >
        0
    ):

        sharpe_ratio = (

            excess_returns.mean()

            /

            excess_std

            *

            np.sqrt(
                TRADING_DAYS
            )
        )

    else:

        sharpe_ratio = np.nan


    # --------------------------------------------------------
    # WEALTH
    # --------------------------------------------------------

    wealth = (

        (
            1
            +
            returns
        )
        .cumprod()
    )


    running_peak = (

        wealth
        .cummax()
    )


    drawdown = (

        wealth

        /

        running_peak

        -

        1
    )


    maximum_drawdown = (

        drawdown.min()
    )


    return pd.Series({

        "Cumulative Return":
            cumulative_return,

        "Annualized Return":
            annualized_return,

        "Annualized Volatility":
            annualized_volatility,

        "Sharpe Ratio":
            sharpe_ratio,

        "Maximum Drawdown":
            maximum_drawdown
    })

In [340]:
# ============================================================
# CELL 10 — OVERALL PERFORMANCE COMPARISON
# ============================================================


# Dynamic benchmark names.
ASSET_BUY_HOLD_LABEL = (
    f"{ASSET_TICKER} Buy & Hold"
)


MARKET_BUY_HOLD_LABEL = (
    f"{MARKET_TICKER} Buy & Hold"
)


# ------------------------------------------------------------
# STRATEGY
# ------------------------------------------------------------

strategy_stats = performance_stats(

    df["Strategy_Return"],

    df["Rf_Daily"]
)


# ------------------------------------------------------------
# SELECTED STOCK BUY & HOLD
# ------------------------------------------------------------

asset_stats = performance_stats(

    df[ASSET_RETURN_COL],

    df["Rf_Daily"]
)


# ------------------------------------------------------------
# SPY
# ------------------------------------------------------------

market_stats = performance_stats(

    df[MARKET_RETURN_COL],

    df["Rf_Daily"]
)


performance_table = pd.DataFrame({

    STRATEGY_LABEL:
        strategy_stats,

    ASSET_BUY_HOLD_LABEL:
        asset_stats,

    MARKET_BUY_HOLD_LABEL:
        market_stats
})


display_table = (
    performance_table.copy()
)


percentage_rows = [

    "Cumulative Return",
    "Annualized Return",
    "Annualized Volatility",
    "Maximum Drawdown"
]


display_table.loc[
    percentage_rows
] = (

    display_table.loc[
        percentage_rows
    ]

    *
    100
)


print(
    "OVERALL PERFORMANCE COMPARISON"
)


print(
    "--------------------------------"
)


print(
    display_table.round(2)
)

OVERALL PERFORMANCE COMPARISON
--------------------------------
                       NVDA Adaptive Alpha Regime Strategy  NVDA Buy & Hold  \
Cumulative Return                                  2819.78          4987.26   
Annualized Return                                    57.65            69.91   
Annualized Volatility                                47.08            51.12   
Sharpe Ratio                                          1.14             1.24   
Maximum Drawdown                                    -68.74           -67.24   

                       SPY Buy & Hold  
Cumulative Return              201.79  
Annualized Return               16.07  
Annualized Volatility           18.55  
Sharpe Ratio                     0.75  
Maximum Drawdown               -32.05  


In [341]:
# ============================================================
# CELL 11 — INTERACTIVE GROWTH OF $1
# ============================================================
#
# IMPORTANT:
#
# The strategy wealth is based on the ACTUAL event-driven
# returns from Cell 7.
#
# Therefore losing positions continue to be marked-to-market
# even though they are not sold.
#
# ============================================================


wealth = pd.DataFrame(
    index=df.index
)


# ------------------------------------------------------------
# STRATEGY WEALTH
# ------------------------------------------------------------

wealth[STRATEGY_LABEL] = (

    (
        1
        +
        df["Strategy_Return"]
    )

    .cumprod()
)


# ------------------------------------------------------------
# SELECTED STOCK BUY & HOLD
# ------------------------------------------------------------

wealth[ASSET_BUY_HOLD_LABEL] = (

    (
        1
        +
        df[ASSET_RETURN_COL]
    )

    .cumprod()
)


# ------------------------------------------------------------
# SPY BUY & HOLD
# ------------------------------------------------------------

wealth[MARKET_BUY_HOLD_LABEL] = (

    (
        1
        +
        df[MARKET_RETURN_COL]
    )

    .cumprod()
)


fig = go.Figure()


# Strategy.
fig.add_trace(
    go.Scatter(

        x=wealth.index,

        y=wealth[
            STRATEGY_LABEL
        ],

        mode="lines",

        name=STRATEGY_LABEL,

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Value: $%{y:.2f}"
            "<extra></extra>"
        )
    )
)


# Stock.
fig.add_trace(
    go.Scatter(

        x=wealth.index,

        y=wealth[
            ASSET_BUY_HOLD_LABEL
        ],

        mode="lines",

        name=ASSET_BUY_HOLD_LABEL,

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Value: $%{y:.2f}"
            "<extra></extra>"
        )
    )
)


# SPY.
fig.add_trace(
    go.Scatter(

        x=wealth.index,

        y=wealth[
            MARKET_BUY_HOLD_LABEL
        ],

        mode="lines",

        name=MARKET_BUY_HOLD_LABEL,

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Value: $%{y:.2f}"
            "<extra></extra>"
        )
    )
)


fig.update_layout(

    title=(
        f"Growth of $1: "
        f"{STRATEGY_LABEL} vs "
        f"{ASSET_BUY_HOLD_LABEL} vs "
        f"{MARKET_TICKER}"
    ),

    xaxis_title="Date",

    yaxis_title="Portfolio Value",

    template="plotly_white",

    hovermode="x unified",

    legend_title_text=""
)


fig.update_yaxes(
    tickprefix="$"
)


fig.show()

In [342]:
# ============================================================
# CELL 12 — ROLLING 1-YEAR PERFORMANCE
# ============================================================


def rolling_max_drawdown(x):


    wealth_window = np.cumprod(
        1
        +
        x
    )


    peak = np.maximum.accumulate(
        wealth_window
    )


    drawdown = (

        wealth_window

        /

        peak

        -

        1
    )


    return np.min(
        drawdown
    )


# Dynamic column names.
STRATEGY_1Y_RETURN_COL = (
    "Strategy_1Y_Return"
)

STRATEGY_1Y_VOL_COL = (
    "Strategy_1Y_Volatility"
)

STRATEGY_1Y_SHARPE_COL = (
    "Strategy_1Y_Sharpe"
)

STRATEGY_1Y_DD_COL = (
    "Strategy_1Y_Max_Drawdown"
)


ASSET_1Y_RETURN_COL = (
    f"{ASSET_TICKER}_1Y_Return"
)

ASSET_1Y_VOL_COL = (
    f"{ASSET_TICKER}_1Y_Volatility"
)

ASSET_1Y_SHARPE_COL = (
    f"{ASSET_TICKER}_1Y_Sharpe"
)

ASSET_1Y_DD_COL = (
    f"{ASSET_TICKER}_1Y_Max_Drawdown"
)


rolling_results = pd.DataFrame(
    index=df.index
)


# ============================================================
# STRATEGY ROLLING RETURN
# ============================================================

rolling_results[
    STRATEGY_1Y_RETURN_COL
] = (

    df["Strategy_Return"]

    .rolling(
        ROLLING_WINDOW
    )

    .apply(

        lambda x:
        np.prod(
            1
            +
            x
        )
        -
        1,

        raw=True
    )
)


# Strategy volatility.
rolling_results[
    STRATEGY_1Y_VOL_COL
] = (

    df["Strategy_Return"]

    .rolling(
        ROLLING_WINDOW
    )

    .std()

    *
    np.sqrt(
        TRADING_DAYS
    )
)


# Strategy excess return.
strategy_excess = (

    df["Strategy_Return"]

    -

    df["Rf_Daily"]
)


# Strategy Sharpe.
rolling_results[
    STRATEGY_1Y_SHARPE_COL
] = (

    strategy_excess

    .rolling(
        ROLLING_WINDOW
    )

    .mean()

    /

    strategy_excess

    .rolling(
        ROLLING_WINDOW
    )

    .std()

    *

    np.sqrt(
        TRADING_DAYS
    )
)


# Strategy drawdown.
rolling_results[
    STRATEGY_1Y_DD_COL
] = (

    df["Strategy_Return"]

    .rolling(
        ROLLING_WINDOW
    )

    .apply(
        rolling_max_drawdown,
        raw=True
    )
)


# ============================================================
# STOCK BUY & HOLD
# ============================================================

rolling_results[
    ASSET_1Y_RETURN_COL
] = (

    df[ASSET_RETURN_COL]

    .rolling(
        ROLLING_WINDOW
    )

    .apply(

        lambda x:
        np.prod(
            1
            +
            x
        )
        -
        1,

        raw=True
    )
)


rolling_results[
    ASSET_1Y_VOL_COL
] = (

    df[ASSET_RETURN_COL]

    .rolling(
        ROLLING_WINDOW
    )

    .std()

    *
    np.sqrt(
        TRADING_DAYS
    )
)


asset_excess = (

    df[ASSET_RETURN_COL]

    -

    df["Rf_Daily"]
)


rolling_results[
    ASSET_1Y_SHARPE_COL
] = (

    asset_excess

    .rolling(
        ROLLING_WINDOW
    )

    .mean()

    /

    asset_excess

    .rolling(
        ROLLING_WINDOW
    )

    .std()

    *

    np.sqrt(
        TRADING_DAYS
    )
)


rolling_results[
    ASSET_1Y_DD_COL
] = (

    df[ASSET_RETURN_COL]

    .rolling(
        ROLLING_WINDOW
    )

    .apply(
        rolling_max_drawdown,
        raw=True
    )
)


rolling_results = (
    rolling_results
    .dropna()
)


print(
    rolling_results.head()
)

            Strategy_1Y_Return  Strategy_1Y_Volatility  Strategy_1Y_Sharpe  \
Date                                                                         
2020-03-31            0.243307                0.467868            0.661338   
2020-04-01            0.189455                0.469981            0.566404   
2020-04-02            0.125618                0.472124            0.449274   
2020-04-03            0.151738                0.473478            0.497975   
2020-04-06            0.145358                0.473397            0.486426   

            Strategy_1Y_Max_Drawdown  NVDA_1Y_Return  NVDA_1Y_Volatility  \
Date                                                                       
2020-03-31                  -0.35985        0.463100            0.484281   
2020-04-01                  -0.35985        0.399932            0.486382   
2020-04-02                  -0.35985        0.324800            0.488529   
2020-04-03                  -0.35985        0.355541            0.489805 

In [343]:
# ============================================================
# CELL 13 — ROLLING 1-YEAR SHARPE
# ============================================================


fig = go.Figure()


fig.add_trace(
    go.Scatter(

        x=rolling_results.index,

        y=rolling_results[
            STRATEGY_1Y_SHARPE_COL
        ],

        mode="lines",

        name=STRATEGY_LABEL
    )
)


fig.add_trace(
    go.Scatter(

        x=rolling_results.index,

        y=rolling_results[
            ASSET_1Y_SHARPE_COL
        ],

        mode="lines",

        name=ASSET_BUY_HOLD_LABEL
    )
)


fig.add_hline(
    y=0,
    line_dash="dash"
)


fig.update_layout(

    title=(
        "Rolling 1-Year Sharpe Ratio: "
        f"{STRATEGY_LABEL} vs "
        f"{ASSET_BUY_HOLD_LABEL}"
    ),

    xaxis_title="Date",

    yaxis_title="Annualized Sharpe Ratio",

    template="plotly_white",

    hovermode="x unified",

    legend_title_text=""
)


fig.show()

In [344]:
# ============================================================
# CELL 14 — ROLLING 1-YEAR RETURN
# ============================================================


fig = go.Figure()


fig.add_trace(
    go.Scatter(

        x=rolling_results.index,

        y=rolling_results[
            STRATEGY_1Y_RETURN_COL
        ],

        mode="lines",

        name=STRATEGY_LABEL,

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "1-Year Return: %{y:.2%}"
            "<extra></extra>"
        )
    )
)


fig.add_trace(
    go.Scatter(

        x=rolling_results.index,

        y=rolling_results[
            ASSET_1Y_RETURN_COL
        ],

        mode="lines",

        name=ASSET_BUY_HOLD_LABEL,

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "1-Year Return: %{y:.2%}"
            "<extra></extra>"
        )
    )
)


fig.add_hline(
    y=0,
    line_dash="dash"
)


fig.update_layout(

    title=(
        "Rolling 1-Year Return: "
        f"{STRATEGY_LABEL} vs "
        f"{ASSET_BUY_HOLD_LABEL}"
    ),

    xaxis_title="Date",

    yaxis_title="1-Year Return",

    template="plotly_white",

    hovermode="x unified",

    legend_title_text=""
)


fig.update_yaxes(
    tickformat=".0%"
)


fig.show()

In [345]:
# ============================================================
# CELL 15 — ROLLING 1-YEAR VOLATILITY
# ============================================================


fig = go.Figure()


fig.add_trace(
    go.Scatter(

        x=rolling_results.index,

        y=rolling_results[
            STRATEGY_1Y_VOL_COL
        ],

        mode="lines",

        name=STRATEGY_LABEL
    )
)


fig.add_trace(
    go.Scatter(

        x=rolling_results.index,

        y=rolling_results[
            ASSET_1Y_VOL_COL
        ],

        mode="lines",

        name=ASSET_BUY_HOLD_LABEL
    )
)


fig.update_layout(

    title=(
        "Rolling 1-Year Volatility: "
        f"{STRATEGY_LABEL} vs "
        f"{ASSET_BUY_HOLD_LABEL}"
    ),

    xaxis_title="Date",

    yaxis_title="Annualized Volatility",

    template="plotly_white",

    hovermode="x unified",

    legend_title_text=""
)


fig.update_yaxes(
    tickformat=".0%"
)


fig.show()

In [346]:
# ============================================================
# CELL 16A — FULL-PERIOD DRAWDOWN
# ============================================================


# Strategy wealth.
strategy_wealth = (

    (
        1
        +
        df["Strategy_Return"]
    )
    .cumprod()
)


strategy_peak = (
    strategy_wealth.cummax()
)


strategy_drawdown = (

    strategy_wealth
    /
    strategy_peak
    -
    1
)


# Buy-and-hold stock wealth.
asset_wealth = (

    (
        1
        +
        df[ASSET_RETURN_COL]
    )
    .cumprod()
)


asset_peak = (
    asset_wealth.cummax()
)


asset_drawdown = (

    asset_wealth
    /
    asset_peak
    -
    1
)


fig = go.Figure()


fig.add_trace(
    go.Scatter(

        x=df.index,

        y=strategy_drawdown,

        mode="lines",

        name=STRATEGY_LABEL,

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Drawdown: %{y:.2%}"
            "<extra></extra>"
        )
    )
)


fig.add_trace(
    go.Scatter(

        x=df.index,

        y=asset_drawdown,

        mode="lines",

        name=ASSET_BUY_HOLD_LABEL,

        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Drawdown: %{y:.2%}"
            "<extra></extra>"
        )
    )
)


fig.add_hline(
    y=0,
    line_dash="dash"
)


fig.update_layout(

    title=(
        "Historical Drawdown: "
        f"{STRATEGY_LABEL} vs "
        f"{ASSET_BUY_HOLD_LABEL}"
    ),

    xaxis_title="Date",

    yaxis_title="Drawdown",

    template="plotly_white",

    hovermode="x unified",

    legend_title_text=""
)


fig.update_yaxes(
    tickformat=".0%"
)


fig.show()

In [347]:
# ============================================================
# CELL 17 — DEFINE CAPM REGRESSION FUNCTION
# ============================================================


def run_capm(
    data,
    period_name
):


    # --------------------------------------------------------
    # REGRESSION DATA
    # --------------------------------------------------------

    regression_data = (

        data[
            [
                "Strategy_Excess",
                "Market_Excess"
            ]
        ]
        .dropna()
    )


    # Require a reasonable minimum sample.
    if len(regression_data) < 30:

        raise ValueError(

            f"{period_name}: "
            "Too few observations for CAPM."
        )


    # --------------------------------------------------------
    # X = MARKET EXCESS RETURN
    # --------------------------------------------------------

    X = regression_data[
        "Market_Excess"
    ]


    # Add intercept:
    #
    # this becomes alpha.
    X = sm.add_constant(
        X
    )


    # --------------------------------------------------------
    # Y = STRATEGY EXCESS RETURN
    # --------------------------------------------------------

    y = regression_data[
        "Strategy_Excess"
    ]


    # --------------------------------------------------------
    # FIT OLS WITH HAC ERRORS
    # --------------------------------------------------------

    model = sm.OLS(

        y,
        X

    ).fit(

        cov_type="HAC",

        cov_kwds={
            "maxlags": 5
        }
    )


    # --------------------------------------------------------
    # ALPHA
    # --------------------------------------------------------

    alpha_daily = (

        model.params[
            "const"
        ]
    )


    alpha_annualized = (

        alpha_daily
        *
        TRADING_DAYS
    )


    # --------------------------------------------------------
    # BETA
    # --------------------------------------------------------

    beta = (

        model.params[
            "Market_Excess"
        ]
    )


    # --------------------------------------------------------
    # P-VALUES
    # --------------------------------------------------------

    alpha_p_value = (

        model.pvalues[
            "const"
        ]
    )


    beta_p_value = (

        model.pvalues[
            "Market_Excess"
        ]
    )


    # --------------------------------------------------------
    # ALPHA T-STAT
    # --------------------------------------------------------

    alpha_t_stat = (

        model.tvalues[
            "const"
        ]
    )


    # --------------------------------------------------------
    # R-SQUARED
    # --------------------------------------------------------

    r_squared = (

        model.rsquared
    )


    # --------------------------------------------------------
    # 95% ALPHA CONFIDENCE INTERVAL
    # --------------------------------------------------------

    alpha_ci = (

        model
        .conf_int()
        .loc[
            "const"
        ]

        *
        TRADING_DAYS
    )


    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print(

        f"\n--- "
        f"{period_name} CAPM Results "
        f"({ASSET_TICKER}) ---"
    )


    print(

        f"Annualized Alpha: "
        f"{alpha_annualized:.2%}"
    )


    print(

        f"Beta: "
        f"{beta:.4f}"
    )


    print(

        f"Alpha t-statistic: "
        f"{alpha_t_stat:.3f}"
    )


    print(

        f"Alpha p-value: "
        f"{alpha_p_value:.4f}"
    )


    print(

        f"Beta p-value: "
        f"{beta_p_value:.4f}"
    )


    print(

        f"R-squared: "
        f"{r_squared:.4f}"
    )


    print(

        "95% Alpha Confidence Interval: "

        f"[{alpha_ci.iloc[0]:.2%}, "

        f"{alpha_ci.iloc[1]:.2%}]"
    )


    # --------------------------------------------------------
    # CONCLUSION
    # --------------------------------------------------------

    if (

        alpha_annualized
        >
        0

        and

        alpha_p_value
        <
        0.05
    ):

        print(

            "Conclusion: Alpha is POSITIVE "
            "and statistically significant."
        )


    elif (

        alpha_annualized
        <
        0

        and

        alpha_p_value
        <
        0.05
    ):

        print(

            "Conclusion: Alpha is statistically "
            "significant but NEGATIVE."
        )


    else:

        print(

            "Conclusion: Alpha is NOT "
            "statistically significant."
        )


    return model

In [348]:
# ============================================================
# CELL 18 — RUN FULL-SAMPLE CAPM + INTERACTIVE CAPM PLOT
# ============================================================


# Run regression.
capm_model = run_capm(

    df,

    "Full Backtest"
)


# ------------------------------------------------------------
# PREPARE GRAPH DATA
# ------------------------------------------------------------

capm_plot_data = (

    df[
        [
            "Strategy_Excess",
            "Market_Excess"
        ]
    ]
    .dropna()
)


# Sort x-values so regression line is smooth.
x_line = (

    capm_plot_data[
        "Market_Excess"
    ]
    .sort_values()
)


# Fitted CAPM line:
#
# y =
# alpha + beta*x
y_line = (

    capm_model.params[
        "const"
    ]

    +

    capm_model.params[
        "Market_Excess"
    ]

    *
    x_line
)


# ------------------------------------------------------------
# CREATE GRAPH
# ------------------------------------------------------------

fig = go.Figure()


# Actual daily observations.
fig.add_trace(

    go.Scatter(

        x=(
            capm_plot_data[
                "Market_Excess"
            ]
            *
            100
        ),

        y=(
            capm_plot_data[
                "Strategy_Excess"
            ]
            *
            100
        ),

        mode="markers",

        name="Daily Observations",

        hovertemplate=(

            "Market Excess Return: "
            "%{x:.2f}%<br>"

            "Strategy Excess Return: "
            "%{y:.2f}%"

            "<extra></extra>"
        )
    )
)


# Regression line.
fig.add_trace(

    go.Scatter(

        x=x_line * 100,

        y=y_line * 100,

        mode="lines",

        name="CAPM Regression Line",

        hovertemplate=(

            "Market Excess Return: "
            "%{x:.2f}%<br>"

            "Predicted Strategy Return: "
            "%{y:.2f}%"

            "<extra></extra>"
        )
    )
)


# ------------------------------------------------------------
# GRAPH FORMAT
# ------------------------------------------------------------

annual_alpha = (

    capm_model.params[
        "const"
    ]
    *
    TRADING_DAYS
)


beta = (

    capm_model.params[
        "Market_Excess"
    ]
)


r_squared = (

    capm_model.rsquared
)


fig.update_layout(

    title=(

        f"CAPM Regression: "
        f"{STRATEGY_LABEL}"
    ),

    xaxis_title=(

        f"{MARKET_TICKER} "
        "Excess Return (%)"
    ),

    yaxis_title=(

        f"{ASSET_TICKER} Strategy "
        "Excess Return (%)"
    ),

    template="plotly_white",

    legend_title_text=""
)


# Display alpha, beta and R² directly on graph.
fig.add_annotation(

    x=0.02,

    y=0.98,

    xref="paper",

    yref="paper",

    xanchor="left",

    yanchor="top",

    text=(

        f"Annual Alpha = "
        f"{annual_alpha:.2%}<br>"

        f"Beta = "
        f"{beta:.4f}<br>"

        f"R² = "
        f"{r_squared:.4f}"
    ),

    showarrow=False
)


fig.show()


--- Full Backtest CAPM Results (NVDA) ---
Annualized Alpha: 30.61%
Beta: 1.6658
Alpha t-statistic: 2.491
Alpha p-value: 0.0128
Beta p-value: 0.0000
R-squared: 0.4311
95% Alpha Confidence Interval: [6.52%, 54.70%]
Conclusion: Alpha is POSITIVE and statistically significant.


In [349]:
# ============================================================
# CELL 19 — DISPLAY FULL REGRESSION OUTPUT
# ============================================================


print(

    capm_model.summary()
)


# ------------------------------------------------------------
# IMPORTANT ROWS
# ------------------------------------------------------------
#
# const
#
# =
# daily CAPM alpha
#
#
# Market_Excess
#
# =
# beta
#
#
# P>|z|
#
# =
# HAC robust p-value
#
#
# [0.025, 0.975]
#
# =
# 95% confidence interval
#
#
# IMPORTANT:
#
# Statistical significance does NOT necessarily mean
# economic significance.

                            OLS Regression Results                            
Dep. Variable:        Strategy_Excess   R-squared:                       0.431
Model:                            OLS   Adj. R-squared:                  0.431
Method:                 Least Squares   F-statistic:                     484.6
Date:                Wed, 09 Sep 2026   Prob (F-statistic):           1.16e-95
Time:                        22:19:40   Log-Likelihood:                 4448.7
No. Observations:                1868   AIC:                            -8893.
Df Residuals:                    1866   BIC:                            -8882.
Df Model:                           1                                         
Covariance Type:                  HAC                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0012      0.000      2.491

In [350]:
# ============================================================
# CELL 20 — SUBPERIOD ROBUSTNESS TEST
# ============================================================
#
# PURPOSE:
#
# Compare:
#
# - earlier sample
# - most recent two years
#
# This helps determine whether estimated alpha is persistent.
# ============================================================


# Exactly two years before END_DATE.
split_date = (

    pd.Timestamp(
        END_DATE
    )

    -

    pd.DateOffset(
        years=2
    )
)


# Earlier observations.
df_earlier = (

    df[
        df.index
        <
        split_date
    ]
)


# Recent two-year observations.
df_recent = (

    df[
        df.index
        >=
        split_date
    ]
)


# Earlier CAPM.
model_earlier = run_capm(

    df_earlier,

    "Earlier Period"
)


# Recent CAPM.
model_recent = run_capm(

    df_recent,

    "Recent 2-Year Period"
)


--- Earlier Period CAPM Results (NVDA) ---
Annualized Alpha: 32.67%
Beta: 1.6916
Alpha t-statistic: 2.110
Alpha p-value: 0.0349
Beta p-value: 0.0000
R-squared: 0.4236
95% Alpha Confidence Interval: [2.32%, 63.03%]
Conclusion: Alpha is POSITIVE and statistically significant.

--- Recent 2-Year Period CAPM Results (NVDA) ---
Annualized Alpha: 25.49%
Beta: 1.5745
Alpha t-statistic: 1.434
Alpha p-value: 0.1515
Beta p-value: 0.0000
R-squared: 0.4671
95% Alpha Confidence Interval: [-9.35%, 60.33%]
Conclusion: Alpha is NOT statistically significant.
